## Udacity Steering Angle Prediction Dataset

The [Udacity Self driving Car](https://github.com/udacity/self-driving-car) is an open source project aimed at developing DL systems for various aspects of a self driving car. The datasets are avaiable to download from the [Github repository](https://github.com/udacity/self-driving-car/tree/master/datasets).




In [1]:
%load_ext autotime 
#to measure cell execution time of all cells

time: 104 μs (started: 2026-03-30 14:40:03 +01:00)


In [2]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from PIL import Image

time: 4.39 s (started: 2026-03-30 14:40:03 +01:00)


#### Parameters and Variables

In [3]:
import os
from pathlib import Path



# Model params
img_x, img_y = 32, 32
resize_shape = (img_x, img_y)
batch_size = 20
num_epochs = 15
num_frames = 30
learning_rate = 1e-4
gamma=0.1
milestones = [0.5 * num_epochs, 0.75 * num_epochs]


# the min angle is -0.4 and the max angle is 0.4, we try to divide it into 3 classes
num_classes = 3 
num_channels = 3 # RGB

# 3D CNN parameters
fc_hidden1, fc_hidden2 = 32, 32
dropout = 0.5       # dropout probability


DEVICE = 'cuda:1' if torch.cuda.is_available() else 'cpu'

# Path
PWD = Path(os.path.abspath('')).parent.resolve()
MODELS_PATH = Path(os.path.abspath('')).parent.resolve() / "models/"
DATASET_PATH = "/vol/bitbucket/svadakku/data/udacity/Ch2_001"
BOUNDING_BOX_PATH = "/vol/bitbucket/svadakku/data/udacity/Ch2_001_tracking/center_tracking"
# TEST_DATASET_PATH = "/vol/bitbucket/svadakku/data/udacity/Challenge2/Test"

time: 28.8 ms (started: 2026-03-30 14:40:07 +01:00)


In [4]:

# Dataset to convert Udacity images into sequence of frames
class UdacitySteeringAngleDataset(Dataset):
    """Custom Dataset for loading images and steering angles for Udacity self-driving car dataset."""
    
    def __init__(self, image_dir, labels_file, transform=None, train=True, test_size=0.2, random_state=42, resize_shape=(img_x, img_y), num_classes=10, num_frames=30):
        """
        Args:
            image_dir (str): Directory containing all the images.
            labels_file (str): Path to CSV file with image filenames and steering angles.
            transform (callable, optional): Optional transform to be applied on a sample.
            train (bool): training or testing split
            test_size (float): Proportion of the dataset to be used as test set.
            random_state (int): Random seed for reproducibility.
        """
        self.image_dir = image_dir
        self.labels_file = labels_file
        
        self.train = train
        self.img_x = img_x
        self.img_y = img_y
        self.num_classes = num_classes
        self.num_frames = num_frames
        self.thresholds = np.linspace(-1.0, 1.0, num_classes + 1)
        
        
        # Define the transform to be applied to the images
        # If transform is not provided, use default transformations
        if transform is not None:
            self.transform = transform
        else:
            train_transform = transforms.Compose([
                transforms.Resize((img_x, img_y)),  # Typical size for Udacity dataset
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Add some color variation
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Small random shifts
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            
            test_transform = transforms.Compose([
                transforms.Resize((img_x, img_y)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            if train:
                # Define transforms for steering angle prediction task
                self.transform = train_transform
            else:
                self.transform = test_transform
                
        
        # Read the CSV file with image filenames and steering angles
        self.data_frame = pd.read_csv(labels_file)
        
        # Get image paths and steering angles
        self.image_files = [os.path.join(image_dir, f"{filename}.jpg") for filename in self.data_frame['frame_id'].values]
        self.steering_angles = self.data_frame['steering_angle'].values.astype(np.float32)
        
        # Split the dataset into train and test sets
        train_files, test_files, train_angles, test_angles = train_test_split(
            self.image_files, self.steering_angles, test_size=test_size, random_state=random_state
        )
        
        # Select the appropriate data split
        if train:
            self.image_files = train_files
            self.steering_angles = train_angles
        else:  # 'test' split
            self.image_files = test_files
            self.steering_angles = test_angles
    
    
    def __len__(self):
        """Return the total number of images."""
        return len(self.image_files)
    
    
    def __getitem__(self, idx):
        
        start_idx = idx
        if (start_idx + self.num_frames  >= len(self.image_files)):
            # If the index is too close to the end, we need to backtrack
            start_idx = len(self.image_files) - self.num_frames - 1
        
        
        # Load images
        images = []
        angle_labels = []
        for i in range(start_idx, start_idx+self.num_frames):
            img_path = self.image_files[idx]
            steering_angle = self.steering_angles[idx]
            img = Image.open(img_path)
            if self.transform:
                img = self.transform(img)
            images.append(img)
            # Get steering angle and convert to class
            angle_labels.append(self._get_steering_angle_class_(steering_angle))
        
        # Stack images into tensor
        image_sequence = torch.stack(images, dim=1) # Shape: (channels, depth, W, H)
        
        
        steering_class = max(set(angle_labels), key=angle_labels.count)
        
        return image_sequence, steering_class


    def _get_steering_angle_class_(self, angle):
        """
        Convert continuous steering angles to discrete classes.
        
        Classes example (10 classes):
        - Extreme Left:  [-1.0 to -0.8]   -> 0
        - Hard Left:     [-0.8 to -0.6]   -> 1
        - Medium Left:   [-0.6 to -0.4]   -> 2
        - Soft Left:     [-0.4 to -0.2]   -> 3
        - Slight Left:   [-0.2 to 0.0]    -> 4
        - Slight Right:  [0.0 to 0.2]     -> 5
        - Soft Right:    [0.2 to 0.4]     -> 6
        - Medium Right:  [0.4 to 0.6]     -> 7
        - Hard Right:    [0.6 to 0.8]     -> 8
        - Extreme Right: [0.8 to 1.0]     -> 9
        
        Classes example (3 classes):
        - Left: [-1.0 to -0.333] -> 0
        - Straight:[-0.333 to 0.3333] -> 1
        - Right: [-0.333 to 1]   -> 2
        """
        angle = np.clip(angle, -1.0, 1.0)  # Ensure angle is within bounds
        for i in range(len(self.thresholds) - 1):
            if self.thresholds[i] <= angle <= self.thresholds[i + 1]:
                return i
        return self.num_classes - 1  # For edge case of angle = 1.0
    
    

# Load training and validation datasets
train_dataset = UdacitySteeringAngleDataset(image_dir=f"{DATASET_PATH}/center", labels_file=f"{DATASET_PATH}/final_example.csv", train=True, resize_shape=resize_shape, num_classes=num_classes, num_frames=num_frames)
val_dataset = UdacitySteeringAngleDataset(image_dir=f"{DATASET_PATH}/center", labels_file=f"{DATASET_PATH}/final_example.csv", train=False, resize_shape=resize_shape, num_classes=num_classes, num_frames=num_frames)

train_dataset_at_eval = UdacitySteeringAngleDataset(image_dir=f"{DATASET_PATH}/center", labels_file=f"{DATASET_PATH}/final_example.csv", train=True, resize_shape=resize_shape, num_classes=num_classes)
# Set the transform of the training eval to be the same as the validation dataset
train_dataset_at_eval.transform = val_dataset.transform

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
train_loader_at_eval = DataLoader(train_dataset_at_eval, batch_size=batch_size, shuffle=False)

time: 113 ms (started: 2026-03-30 14:40:07 +01:00)


#### 3DCNN Model

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def conv3D_output_size(img_size, padding, kernel_size, stride):
    # compute output shape of conv3D
    outshape = (np.floor((img_size[0] + 2 * padding[0] - (kernel_size[0] - 1) - 1) / stride[0] + 1).astype(int),
                np.floor((img_size[1] + 2 * padding[1] - (kernel_size[1] - 1) - 1) / stride[1] + 1).astype(int),
                np.floor((img_size[2] + 2 * padding[2] - (kernel_size[2] - 1) - 1) / stride[2] + 1).astype(int))
    return outshape

class CNN3D(nn.Module):
    def __init__(self, t_dim=120, img_x=90, img_y=120, drop_p=0.2, fc_hidden1=256, fc_hidden2=128, num_classes=50, in_channels=1):
        super(CNN3D, self).__init__()
        
        self.layers = []
        self.batch_norms = []        
        # set video dimension
        self.t_dim = t_dim
        self.img_x = img_x
        self.img_y = img_y
        # fully connected layer hidden nodes
        self.fc_hidden1, self.fc_hidden2 = fc_hidden1, fc_hidden2
        self.drop_p = drop_p
        self.num_classes = num_classes
        self.ch1, self.ch2 = 32, 48
        self.k1, self.k2 = (5, 5, 5), (3, 3, 3)  # 3d kernel size
        self.s1, self.s2 = (2, 2, 2), (2, 2, 2)  # 3d strides
        self.pd1, self.pd2 = (0, 0, 0), (0, 0, 0)  # 3d padding

        # compute conv1 & conv2 output shape
        self.conv1_outshape = conv3D_output_size((self.t_dim, self.img_x, self.img_y), self.pd1, self.k1, self.s1)
        self.conv2_outshape = conv3D_output_size(self.conv1_outshape, self.pd2, self.k2, self.s2)

        self.conv1 = nn.Conv3d(in_channels=in_channels, out_channels=self.ch1, kernel_size=self.k1, stride=self.s1,
                               padding=self.pd1)
        self.layers.append("Conv3d_2_0")
        
        self.bn1 = nn.BatchNorm3d(self.ch1)
        self.layers.append("BatchNorm3d_32")
        self.batch_norms.append(self.bn1)
        
        self.layers.append("ReLU")
        
        self.conv2 = nn.Conv3d(in_channels=self.ch1, out_channels=self.ch2, kernel_size=self.k2, stride=self.s2,
                               padding=self.pd2)
        self.layers.append("Conv3d_2_0")
        
        self.bn2 = nn.BatchNorm3d(self.ch2)
        self.layers.append("BatchNorm3d_48")
        self.batch_norms.append(self.bn2)
        
        self.relu = nn.ReLU(inplace=True)
        self.layers.append("ReLU")
        
        self.drop = nn.Dropout3d(self.drop_p)
        self.pool = nn.MaxPool3d(2)
        
        # Add adaptive pooling layer to handle different input sizes
        self.adaptive_pool = nn.AdaptiveAvgPool3d((2, 2, 2))
        
        self.layers.append("Flatten")
        # Fixed size output from adaptive pooling: ch2 * 2 * 2 * 2
        fc_input_size = self.ch2 * 2 * 2 * 2
        
        self.fc1 = nn.Linear(fc_input_size, self.fc_hidden1)
        self.layers.append("Linear")
        self.layers.append("ReLU")
        
        self.fc2 = nn.Linear(self.fc_hidden1, self.fc_hidden2)
        self.layers.append("Linear")
        self.layers.append("ReLU")
        
        self.fc3 = nn.Linear(self.fc_hidden2, self.num_classes)  # fully connected layer, output = multi-classes
        self.layers.append("Linear")
        
    def forward(self, x_3d):
        # Conv 1
        x = self.conv1(x_3d)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.drop(x)
        
        # Conv 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.drop(x)
        
        # Use adaptive pooling to get fixed size output regardless of input dimensions
        x = self.adaptive_pool(x)
        
        # FC 1 and 2
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=self.drop_p, training=self.training)
        x = self.fc3(x)
        
        return x 
    

class UdacitySteeringAngle_3DCNN(nn.Module):
    def __init__(self, num_classes=3, in_channels=3):
        super(UdacitySteeringAngle_3DCNN, self).__init__()
        self.num_classes = num_classes
        self.num_frames = num_frames
        self.img_x = img_x
        self.img_y = img_y
        
        # Define the 3D CNN model
        self.cnn3d = CNN3D(t_dim=num_frames, img_x=img_x, img_y=img_y, fc_hidden1=fc_hidden1, fc_hidden2=fc_hidden2, num_classes=num_classes, in_channels=in_channels, drop_p=dropout)
        
    def forward(self, x):
        return self.cnn3d(x)

time: 5.45 ms (started: 2026-03-30 14:40:07 +01:00)


In [6]:
from copy import deepcopy
from collections import OrderedDict
from tqdm import trange

model_path = MODELS_PATH / "udacity_3dcnn_32_32_30.pth"
device = torch.device(DEVICE)

model = UdacitySteeringAngle_3DCNN(num_classes=num_classes, in_channels=num_channels).to(device)
model.load_state_dict(torch.load(model_path))
model = model.eval()

time: 334 ms (started: 2026-03-30 14:40:07 +01:00)


## CROWN-IBP

In [7]:
import sys
from copy import deepcopy
from pathlib import Path
# Add path to find the custom operator
project_root = Path('/data/sv/temporal-nn-verification/')
sys.path.append(str(project_root / 'ucf101_classification'))
sys.path.append(str(project_root / 'ucf101_classification/utils'))

from auto_LiRPA import BoundedModule, BoundedTensor, register_custom_op
from auto_LiRPA.perturbations import PerturbationLpNorm
from auto_LiRPA_custom_operators.convolution_3d import BoundConv3D
from auto_LiRPA_custom_operators.pooling_3d import BoundAveragePoolGeneral

# Register custom operator for 3D convolutions to avoid Conv2D type errors
register_custom_op("onnx::Conv", BoundConv3D)
register_custom_op("onnx::AveragePool", BoundAveragePoolGeneral)
# custom_operators = {
#     'AdaptiveAvgPool3d': BoundAdaptiveAvgPool3d
# }

crown_rob_acc = []
crown_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        if idx == 0:
            lirpa_model = BoundedModule(
                deepcopy(model),
                torch.empty_like(X),
                # custom_ops = custom_operators,
                device=X.device,
                verbose=False,
            )
        
        y = y.to(DEVICE).to(torch.long)
        ptb = PerturbationLpNorm(norm=torch.inf, eps=eps)
        ptb_inputs = BoundedTensor(X, ptb)
        
        final_lower, final_upper = lirpa_model.compute_bounds(x=(ptb_inputs,), method='CROWN-IBP')
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    
    crown_acc.append((correct / total) * 100)
    crown_rob_acc.append((robust / total) * 100)

print("Model Accuracy: ", crown_acc)
print("CROWN-IBP Robust Accuracy: ", crown_rob_acc)


Model Accuracy:  [84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579]
CROWN-IBP Robust Accuracy:  [0.0, 0.0, 5.6099732858414955, 84.23864648263579, 84.23864648263579]
time: 9min 7s (started: 2026-03-26 16:17:39 +00:00)


#### IBP Verification

In [7]:
import random
import torch

# Set random seed for reproducibility
random.seed(42)
torch.manual_seed(42)

# Get a list of indices for random sampling
val_dataset_size = len(val_dataset)
sample_size = 50
sample_indices = random.sample(range(val_dataset_size), min(sample_size, val_dataset_size))

# Create a subset of the validation dataset
val_subset = torch.utils.data.Subset(val_dataset, sample_indices)

# Create a new dataloader for the subset
valid_loader = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False)

time: 3.42 ms (started: 2026-01-20 03:19:47 +00:00)


In [8]:
from typing import *
import torch.nn.functional as F
import torch.nn as nn

def propagate_bounds(input_bounds: Tuple[torch.Tensor, torch.Tensor], layer: nn.Module
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Propagate bounds through a single layer using IBP.

        Args:
            input_bounds: Tuple of (lower_bounds, upper_bounds) for the input
            layer: The layer to propagate bounds through

        Returns:
            Tuple of (lower_bounds, upper_bounds) for the output
        """
        lower_bounds, upper_bounds = input_bounds

        if isinstance(layer, nn.Linear):
            return _propagate_linear(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.Conv3d):
            return _propagate_conv3d(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.ReLU):
            return _propagate_relu(lower_bounds, upper_bounds)
        elif isinstance(layer, nn.BatchNorm3d) or isinstance(layer, nn.BatchNorm2d):
            return _propagate_batchnorm(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.Dropout):
            return _propagate_dropout(lower_bounds, upper_bounds, layer)
        elif isinstance(layer, nn.AdaptiveAvgPool3d):
            return _propagate_adaptive_avgpool3d(
                lower_bounds, upper_bounds, layer.output_size
            )
        else:
            raise NotImplementedError(
                f"IBP propagation not implemented for {type(layer)}"
            )

# Convolutional layer IBP
def _propagate_conv3d(lower_bound, upper_bound, conv_layer):
    # Compute center and radius
    x_mu = (lower_bound + upper_bound) / 2
    x_r = (upper_bound - lower_bound) / 2

    # Compute bounds
    W = conv_layer.weight
    abs_W = W.abs()

    conv0 = F.conv3d(
        x_mu,
        W,
        bias=conv_layer.bias,
        stride=conv_layer.stride,
        padding=conv_layer.padding,
        dilation=conv_layer.dilation,
        groups=conv_layer.groups,
    )
    conv1 = F.conv3d(
        x_r,
        abs_W,
        bias=None,
        stride=conv_layer.stride,
        padding=conv_layer.padding,
        dilation=conv_layer.dilation,
        groups=conv_layer.groups,
    )

    z_L = conv0 - conv1
    z_U = conv0 + conv1

    return z_L, z_U

def _propagate_linear(lower_bound, upper_bound, fc_layer):
    # Compute center and radius
    x_mu = (lower_bound + upper_bound) / 2
    x_r = (upper_bound - lower_bound) / 2

    # Compute bounds
    W = fc_layer.weight
    abs_W = W.abs()
    z_L = F.linear(x_mu, W, bias=fc_layer.bias) - F.linear(x_r, abs_W, bias=None)
    z_U = F.linear(x_mu, W, bias=fc_layer.bias) + F.linear(x_r, abs_W, bias=None)

    return z_L, z_U

def _propagate_relu(lower_bound, upper_bound):
    return F.relu(lower_bound), F.relu(upper_bound)

def _propagate_batchnorm(lower_bound, upper_bound, bn_layer):
    running_mean = bn_layer.running_mean
    running_var = bn_layer.running_var

    W = bn_layer.state_dict()["weight"]
    b = bn_layer.state_dict()["bias"]

    # Compute center and radius
    x_mu = (lower_bound + upper_bound) / 2
    x_r = (upper_bound - lower_bound) / 2

    lower_offset = F.batch_norm(
        -x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )
    upper_offset = F.batch_norm(
        x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )

    lower_offset = torch.minimum(lower_offset, upper_offset)
    upper_offset = torch.maximum(lower_offset, upper_offset)

    nominal = F.batch_norm(x_mu, running_mean, running_var, W, b)

    y_l = nominal + lower_offset
    y_u = nominal + upper_offset
    return y_l, y_u

def _propagate_adaptive_avgpool3d(lower_bound, upper_bound, output_size):
    return F.adaptive_avg_pool3d(lower_bound, output_size), F.adaptive_avg_pool3d(
        upper_bound, output_size
    )

def _propagate_dropout(lower_bound, upper_bound, dropout_layer):
    """
    Propagate bounds through a dropout layer.
    
    During verification, we assume the model is in evaluation mode,
    so dropout should effectively be an identity function.
    
    Args:
        lower_bound: Lower bounds of the input
        upper_bound: Upper bounds of the input
        dropout_layer: The dropout layer to propagate through
        
    Returns:
        Tuple of (lower_bounds, upper_bounds) for the output
    """
    if model.training:
        # Only apply scaling during training
        # During training, dropout scales values by 1/(1-p)
        scale = 1.0 / (1.0 - dropout_layer.p)
        return lower_bound * scale, upper_bound * scale
    else:
        # During evaluation, dropout is an identity function
        return lower_bound, upper_bound


def _propagate_conv3d(
        lower_bounds: torch.Tensor, upper_bounds: torch.Tensor, layer: nn.Conv3d
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Propagate bounds through a 3D convolutional layer"""
        # Compute center and radius
        x_mu = (lower_bounds + upper_bounds) / 2
        x_r = (upper_bounds - lower_bounds) / 2

        # Compute bounds
        W = layer.weight
        abs_W = W.abs()

        conv0 = F.conv3d(
            x_mu,
            W,
            bias=layer.bias,
            stride=layer.stride,
            padding=layer.padding,
            dilation=layer.dilation,
            groups=layer.groups,
        )
        conv1 = F.conv3d(
            x_r,
            abs_W,
            bias=None,
            stride=layer.stride,
            padding=layer.padding,
            dilation=layer.dilation,
            groups=layer.groups,
        )

        z_L = conv0 - conv1
        z_U = conv0 + conv1

        return z_L, z_U

def _propagate_batchnorm3d(
    lower_bounds: torch.Tensor,
    upper_bounds: torch.Tensor,
    layer: nn.BatchNorm3d,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Propagate bounds through a 3D batch normalization layer"""
    running_mean = layer.running_mean
    running_var = layer.running_var

    W = layer.state_dict()["weight"]
    b = layer.state_dict()["bias"]

    # Compute center and radius
    x_mu = (lower_bounds + upper_bounds) / 2
    x_r = (upper_bounds - lower_bounds) / 2

    lower_offset = F.batch_norm(
        -x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )
    upper_offset = F.batch_norm(
        x_r, torch.zeros_like(running_mean), running_var, W, torch.zeros_like(b)
    )

    lower_offset = torch.minimum(lower_offset, upper_offset)
    upper_offset = torch.maximum(lower_offset, upper_offset)

    nominal = F.batch_norm(x_mu, running_mean, running_var, W, b)

    y_l = nominal + lower_offset
    y_u = nominal + upper_offset
    return y_l, y_u

def _propagate_adaptive_avg_pool3d(
    lower_bounds: torch.Tensor,
    upper_bounds: torch.Tensor,
    output_size: Tuple[int, int, int],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Propagate bounds through an adaptive average pooling layer"""
    return (
        F.adaptive_avg_pool3d(lower_bounds, output_size),
        F.adaptive_avg_pool3d(upper_bounds, output_size),
    )
    
    


time: 3.81 ms (started: 2026-01-20 03:19:47 +00:00)


In [9]:
from typing import Optional, Tuple
import time

# Interval bound propagation through the CNN3D model
def ibp_propagate_bounds_cnn3d(
    input_tensor: torch.Tensor,
    epsilon: float = 0.1,
    lower_input: Optional[torch.Tensor] = None,
    upper_input: Optional[torch.Tensor] = None,
    skip_first_conv: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Propagate bounds through the entire UdacitySteeringAngle_3DCNN model"""
    # Set initial bounds
    if lower_input is not None and upper_input is not None:
        lower_bounds, upper_bounds = lower_input, upper_input
    else:
        lower_bounds = input_tensor - epsilon
        upper_bounds = input_tensor + epsilon

    # Verify model instance is a CNN3D or a subclass of it
    assert isinstance(
        model, CNN3D
    ), f"Model is not an instance of CNN3D but {type(model).__name__}"

    # Process convolutional layers
    # Conv layer 1
    if not skip_first_conv:
        lower_bounds, upper_bounds = _propagate_conv3d(
            lower_bounds, upper_bounds, model.conv1
        )
    
    
    lower_bounds, upper_bounds = _propagate_batchnorm3d(
        lower_bounds, upper_bounds, model.bn1
    )
    lower_bounds, upper_bounds = _propagate_relu(lower_bounds, upper_bounds)
    
    
    # Conv layer 2
    lower_bounds, upper_bounds = _propagate_conv3d(
        lower_bounds, upper_bounds, model.conv2
    )
    lower_bounds, upper_bounds = _propagate_batchnorm3d(
        lower_bounds, upper_bounds, model.bn2
    )
    lower_bounds, upper_bounds = _propagate_relu(lower_bounds, upper_bounds)
    
    lower_bounds, upper_bounds = _propagate_adaptive_avg_pool3d(lower_bounds, upper_bounds, (2, 2, 2))
    
    # Flatten
    if lower_bounds.ndim > 2:
        lower_bounds = lower_bounds.view(lower_bounds.size(0), -1)
        upper_bounds = upper_bounds.view(upper_bounds.size(0), -1)
        
    # FC layer 1
    lower_bounds, upper_bounds = _propagate_linear(
        lower_bounds, upper_bounds, model.fc1
    )
    lower_bounds, upper_bounds = _propagate_relu(lower_bounds, upper_bounds)
    
    # FC layer 2
    lower_bounds, upper_bounds = _propagate_linear(
        lower_bounds, upper_bounds, model.fc2
    )
    lower_bounds, upper_bounds = _propagate_relu(lower_bounds, upper_bounds)
    
    # Dropout (during evaluation, it's just identity)
    # We use our custom dropout propagation function
    # lower_bounds, upper_bounds = _propagate_dropout(
    #     lower_bounds, upper_bounds, model.dropout
    # )
    
    # FC layer 3
    lower_bounds, upper_bounds = _propagate_linear(
        lower_bounds, upper_bounds, model.fc3
    )

    return lower_bounds, upper_bounds

time: 1.87 ms (started: 2026-01-20 03:19:47 +00:00)


In [7]:
model = model.cnn3d.to(device)
model.eval()

CNN3D(
  (conv1): Conv3d(3, 32, kernel_size=(5, 5, 5), stride=(2, 2, 2))
  (bn1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv3d(32, 48, kernel_size=(3, 3, 3), stride=(2, 2, 2))
  (bn2): BatchNorm3d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (drop): Dropout3d(p=0.5, inplace=False)
  (pool): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (adaptive_pool): AdaptiveAvgPool3d(output_size=(2, 2, 2))
  (fc1): Linear(in_features=384, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=3, bias=True)
)

time: 6.28 ms (started: 2026-03-30 14:40:19 +01:00)


In [11]:

start_time = time.time()
ibp_rob_acc = []
epsilons = [0.00001, 0.0001, 0.001, 0.01, 0.1]
for epsilon in epsilons:
    for batch_idx, (x, y) in enumerate(valid_loader):  
        x = x.to(device)
        y = y.to(device)
        verified_correct = 0
        total = x.size(0)

        # print(f"\nIBP Verification with epsilon={epsilon}")
        # print(f"Input shape: {x.shape}")
        # print(f"Number of samples: {total}")

        for i in range(total):
            # print(f"\nVerifying sample {i+1}/{total}")
            input_tensor = x[i:i+1]
            y_i = y[i:i+1]
        
            with torch.no_grad():
                lower_bounds, upper_bounds = ibp_propagate_bounds_cnn3d(
                        input_tensor, epsilon=epsilon
                    )
            
            # Print bound statistics
            # print(f"Lower bounds range: [{lower_bounds.min().item():.4f}, {lower_bounds.max().item():.4f}]")
            # print(f"Upper bounds range: [{upper_bounds.min().item():.4f}, {upper_bounds.max().item():.4f}]")
            # print(f"Bound width: {(upper_bounds - lower_bounds).mean().item():.4f}")

            # Check robustness
            margin = lower_bounds[0, y_i] - torch.max(
                torch.cat([upper_bounds[0, :y_i], upper_bounds[0, y_i + 1 :]], dim=0)
            )

            if margin.item() > 0:
                verified_correct += 1
                # print(f"Sample {i+1} is robust (margin: {margin.item():.4f})")
            # else:
                # print(f"Sample {i+1} is not robust (margin: {margin.item():.4f})")

    ibp_rob_acc.append(verified_correct / total)
print(f"Total time taken: {time.time() - start_time:.2f} seconds")

Total time taken: 31.96 seconds
time: 32 s (started: 2026-01-20 03:19:47 +00:00)


In [ ]:
ibp_rob_acc

[0.75, 0.7, 0.0, 0.0, 0.0]

time: 2.37 ms (started: 2026-01-20 03:20:26 +00:00)


#### Selective Interval Bound Propagation Verification

#### Closed form solution Common functions

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Tuple, List, Dict, Optional, Union
from dataclasses import dataclass
import time
from tqdm import tqdm

@dataclass
class LayerBounds:
    """
    Container for layer activation bounds with utility properties.
    
    Attributes:
        lower: Lower bounds tensor
        upper: Upper bounds tensor
    """
    lower: torch.Tensor
    upper: torch.Tensor
    
    @property
    def center(self) -> torch.Tensor:
        """Compute center of bounds: (L + U) / 2"""
        return (self.lower + self.upper) / 2
    
    @property
    def radius(self) -> torch.Tensor:
        """Compute half-width of bounds: (U - L) / 2"""
        return (self.upper - self.lower) / 2
    
    @property
    def width(self) -> torch.Tensor:
        """Compute total width of bounds: U - L"""
        return self.upper - self.lower
    
    def to(self, device: str) -> 'LayerBounds':
        """Move bounds to specified device."""
        return LayerBounds(
            lower=self.lower.to(device),
            upper=self.upper.to(device)
        )

time: 2.28 ms (started: 2026-03-30 14:40:24 +01:00)


In [9]:
def closed_form_conv3d_bounds(
    input_tensor: torch.Tensor,
    conv_layer: nn.Conv3d,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    bn_layer: Optional[nn.BatchNorm3d] = None,
    device: str = DEVICE
) -> LayerBounds:
    """
    Compute EXACT bounds on first Conv3D layer output using closed-form solution.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    For Conv3D with (optionally folded) weight W and structured perturbations:
    
    Output at position (o, d', h', w'):
        z[o,d',h',w'] = Σ W[o,i,kd,kh,kw] × x[i, ...] + b[o]
    
    With shared temporal perturbation δ_t = δ_t' for all t, t':
        z = z_clean + Σ W_eff × δ
    
    Where W_eff = Σ_kd W[o,c,kd,h,w] (sum over temporal kernel dim)
    
    For adversarial patches with spatial mask M:
        z = z_clean + Σ W_eff × (M ⊙ δ)
    
    Optimal perturbation:
        δ* = ε × sign(W_eff)
    
    Exact bounds:
        upper = z_clean + ε × Σ|W_eff ⊙ M|
        lower = z_clean - ε × Σ|W_eff ⊙ M|
    
    ============================================================================
    
    Args:
        input_tensor: Input tensor (C, D, H, W) or (B, C, D, H, W)
        conv_layer: First Conv3d layer
        epsilon: Perturbation budget (L∞ norm)
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask (H, W) for patch perturbations
        bn_layer: Optional BatchNorm3d to fold into conv
        device: Computation device
        
    Returns:
        LayerBounds containing exact lower and upper bounds
    """
    # Handle input dimensions
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    B, C, D, H, W = input_tensor.shape
    input_tensor = input_tensor.to(device)
    
    # Get layer parameters (with optional BN folding)
    if bn_layer is not None:
        weights, bias = get_folded_conv_params(conv_layer, bn_layer, device)
    else:
        weights = conv_layer.weight.to(device)
        bias = conv_layer.bias.to(device) if conv_layer.bias is not None else None
    
    out_c, in_c, kd, kh, kw = weights.shape
    
    # Get conv parameters
    stride = conv_layer.stride
    padding = conv_layer.padding
    
    if isinstance(stride, int):
        stride = (stride, stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding, padding)
    
    # Compute clean output
    z_clean = F.conv3d(input_tensor, weights, bias, stride=stride, padding=padding)
    
    # Compute output spatial dimensions
    out_D = (D + 2 * padding[0] - kd) // stride[0] + 1
    out_H = (H + 2 * padding[1] - kh) // stride[1] + 1
    out_W = (W + 2 * padding[2] - kw) // stride[2] + 1
    
    # Create spatial mask if needed
    if spatial_mask is None:
        spatial_mask = torch.ones(H, W, device=device)
    else:
        spatial_mask = spatial_mask.to(device)
    
    # Check if all frames share the same perturbation
    all_shared = (len(shared_frames) == 1 and len(shared_frames[0]) == D)
    
    if all_shared:
        # =================================================================
        # OPTIMIZED PATH: All frames share same perturbation
        # =================================================================
        # Sum weights over temporal kernel dimension
        w_temporal_sum = weights.sum(dim=2)  # (out_c, in_c, kh, kw)
        
        # Create masked perturbation tensor
        # Shape: (B, in_c, H, W)
        spatial_pert = spatial_mask.unsqueeze(0).unsqueeze(0).expand(B, in_c, -1, -1) * epsilon
        
        # Compute radius via 2D convolution with absolute weights
        # Apply spatial stride and padding
        radius_2d = F.conv2d(
            spatial_pert,
            w_temporal_sum.abs(),
            bias=None,
            stride=(stride[1], stride[2]),
            padding=(padding[1], padding[2])
        )  # Shape: (B, out_c, out_H, out_W)
        
        # Expand to match output shape
        radius = radius_2d.unsqueeze(2).expand(-1, -1, out_D, -1, -1)
        
    else:
        # =================================================================
        # GENERAL PATH: Handle arbitrary shared frame configurations
        # =================================================================
        radius = _compute_general_closed_form_radius(
            input_tensor.shape,
            weights,
            epsilon,
            shared_frames,
            spatial_mask,
            conv_layer,
            device
        )
    
    # Compute final bounds
    lower_bounds = z_clean - radius
    upper_bounds = z_clean + radius
    
    return LayerBounds(lower=lower_bounds, upper=upper_bounds)


# =============================================================================
# BatchNorm Folding Utilities
# =============================================================================

def fold_batchnorm_into_conv(
    conv_weight: torch.Tensor,
    conv_bias: Optional[torch.Tensor],
    bn_weight: torch.Tensor,
    bn_bias: torch.Tensor,
    bn_mean: torch.Tensor,
    bn_var: torch.Tensor,
    bn_eps: float = 1e-5
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Fold BatchNorm parameters into Conv weights and bias.
    
    BN(x) = γ × (x - μ) / √(σ² + ε) + β
    
    After folding:
        W_folded = W × γ / √(σ² + ε)
        b_folded = (b - μ) × γ / √(σ² + ε) + β
    
    Args:
        conv_weight: Conv weight tensor (out_c, in_c, kd, kh, kw)
        conv_bias: Conv bias tensor (out_c,) or None
        bn_weight: BatchNorm gamma (out_c,)
        bn_bias: BatchNorm beta (out_c,)
        bn_mean: BatchNorm running mean (out_c,)
        bn_var: BatchNorm running variance (out_c,)
        bn_eps: BatchNorm epsilon for numerical stability
        
    Returns:
        (folded_weight, folded_bias)
    """
    # Compute scale factor: γ / √(σ² + ε)
    std = torch.sqrt(bn_var + bn_eps)
    scale = bn_weight / std
    
    # Fold into weights: reshape scale for broadcasting
    # Weight shape: (out_c, in_c, kd, kh, kw)
    weight_shape = [conv_weight.shape[0]] + [1] * (conv_weight.dim() - 1)
    scale_reshaped = scale.view(*weight_shape)
    
    folded_weight = conv_weight * scale_reshaped
    
    # Fold into bias
    if conv_bias is None:
        conv_bias = torch.zeros(conv_weight.shape[0], device=conv_weight.device)
    
    folded_bias = (conv_bias - bn_mean) * scale + bn_bias
    
    return folded_weight, folded_bias


def get_folded_conv_params(
    conv_layer: nn.Conv3d,
    bn_layer: nn.BatchNorm3d,
    device: str = DEVICE
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Get Conv3d parameters with BatchNorm folded in.
    
    Args:
        conv_layer: Conv3d layer
        bn_layer: BatchNorm3d layer (must be in eval mode)
        device: Computation device
        
    Returns:
        (folded_weight, folded_bias)
    """
    conv_weight = conv_layer.weight.to(device)
    conv_bias = conv_layer.bias.to(device) if conv_layer.bias is not None else None
    
    bn_weight = bn_layer.weight.to(device)
    bn_bias = bn_layer.bias.to(device)
    bn_mean = bn_layer.running_mean.to(device)
    bn_var = bn_layer.running_var.to(device)
    bn_eps = bn_layer.eps
    
    return fold_batchnorm_into_conv(
        conv_weight, conv_bias, bn_weight, bn_bias, bn_mean, bn_var, bn_eps
    )


# =============================================================================
# Closed-Form First Layer Bounds (EXACT)
# =============================================================================

def closed_form_conv3d_bounds(
    input_tensor: torch.Tensor,
    conv_layer: nn.Conv3d,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    bn_layer: Optional[nn.BatchNorm3d] = None,
    device: str = DEVICE
) -> LayerBounds:
    """
    Compute EXACT bounds on first Conv3D layer output using closed-form solution.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    For Conv3D with (optionally folded) weight W and structured perturbations:
    
    Output at position (o, d', h', w'):
        z[o,d',h',w'] = Σ W[o,i,kd,kh,kw] × x[i, ...] + b[o]
    
    With shared temporal perturbation δ_t = δ_t' for all t, t':
        z = z_clean + Σ W_eff × δ
    
    Where W_eff = Σ_kd W[o,c,kd,h,w] (sum over temporal kernel dim)
    
    For adversarial patches with spatial mask M:
        z = z_clean + Σ W_eff × (M ⊙ δ)
    
    Optimal perturbation:
        δ* = ε × sign(W_eff)
    
    Exact bounds:
        upper = z_clean + ε × Σ|W_eff ⊙ M|
        lower = z_clean - ε × Σ|W_eff ⊙ M|
    
    ============================================================================
    
    Args:
        input_tensor: Input tensor (C, D, H, W) or (B, C, D, H, W)
        conv_layer: First Conv3d layer
        epsilon: Perturbation budget (L∞ norm)
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask (H, W) for patch perturbations
        bn_layer: Optional BatchNorm3d to fold into conv
        device: Computation device
        
    Returns:
        LayerBounds containing exact lower and upper bounds
    """
    # Handle input dimensions
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    B, C, D, H, W = input_tensor.shape
    input_tensor = input_tensor.to(device)
    
    # Get layer parameters (with optional BN folding)
    if bn_layer is not None:
        weights, bias = get_folded_conv_params(conv_layer, bn_layer, device)
    else:
        weights = conv_layer.weight.to(device)
        bias = conv_layer.bias.to(device) if conv_layer.bias is not None else None
    
    out_c, in_c, kd, kh, kw = weights.shape
    
    # Get conv parameters
    stride = conv_layer.stride
    padding = conv_layer.padding
    
    if isinstance(stride, int):
        stride = (stride, stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding, padding)
    
    # Compute clean output
    z_clean = F.conv3d(input_tensor, weights, bias, stride=stride, padding=padding)
    
    # Compute output spatial dimensions
    out_D = (D + 2 * padding[0] - kd) // stride[0] + 1
    out_H = (H + 2 * padding[1] - kh) // stride[1] + 1
    out_W = (W + 2 * padding[2] - kw) // stride[2] + 1
    
    # Create spatial mask if needed
    if spatial_mask is None:
        spatial_mask = torch.ones(H, W, device=device)
    else:
        spatial_mask = spatial_mask.to(device)
    
    # Check if all frames share the same perturbation
    all_shared = (len(shared_frames) == 1 and len(shared_frames[0]) == D)
    
    if all_shared:
        # =================================================================
        # OPTIMIZED PATH: All frames share same perturbation
        # =================================================================
        # Sum weights over temporal kernel dimension
        w_temporal_sum = weights.sum(dim=2)  # (out_c, in_c, kh, kw)
        
        # Create masked perturbation tensor
        # Shape: (B, in_c, H, W)
        spatial_pert = spatial_mask.unsqueeze(0).unsqueeze(0).expand(B, in_c, -1, -1) * epsilon
        
        # Compute radius via 2D convolution with absolute weights
        # Apply spatial stride and padding
        radius_2d = F.conv2d(
            spatial_pert,
            w_temporal_sum.abs(),
            bias=None,
            stride=(stride[1], stride[2]),
            padding=(padding[1], padding[2])
        )  # Shape: (B, out_c, out_H, out_W)
        
        # Expand to match output shape
        radius = radius_2d.unsqueeze(2).expand(-1, -1, out_D, -1, -1)
        
    else:
        # =================================================================
        # GENERAL PATH: Handle arbitrary shared frame configurations
        # =================================================================
        radius = _compute_general_closed_form_radius(
            input_tensor.shape,
            weights,
            epsilon,
            shared_frames,
            spatial_mask,
            conv_layer,
            device
        )
    
    # Compute final bounds
    lower_bounds = z_clean - radius
    upper_bounds = z_clean + radius
    
    return LayerBounds(lower=lower_bounds, upper=upper_bounds)


def _compute_general_closed_form_radius(
    input_shape: Tuple[int, ...],
    weights: torch.Tensor,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: torch.Tensor,
    conv_layer: nn.Conv3d,
    device: str
) -> torch.Tensor:
    """
    Compute closed-form radius for general (non-all-shared) frame configurations.
    
    For each shared group g and each output position:
        radius_contribution = ε × |Σ_{frames f ∈ g, kernel k hitting f} W[k] ⊙ M|
    
    Total radius = Σ_g radius_contribution_g
    """
    B, C, D, H, W = input_shape
    out_c, in_c, kd, kh, kw = weights.shape
    
    stride = conv_layer.stride
    padding = conv_layer.padding
    
    if isinstance(stride, int):
        stride = (stride, stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding, padding)
    
    out_D = (D + 2 * padding[0] - kd) // stride[0] + 1
    out_H = (H + 2 * padding[1] - kh) // stride[1] + 1
    out_W = (W + 2 * padding[2] - kw) // stride[2] + 1
    
    # Initialize radius tensor
    radius = torch.zeros(B, out_c, out_D, out_H, out_W, device=device)
    
    # For each shared group, compute its contribution
    for g_idx, group in enumerate(shared_frames):
        group_set = set(group)
        
        # For each output temporal position
        for od in range(out_D):
            # Build mask for which kernel depth positions contribute
            kd_mask = torch.zeros(kd, device=device)
            for kd_idx in range(kd):
                input_d = od * stride[0] + kd_idx - padding[0]
                if 0 <= input_d < D and input_d in group_set:
                    kd_mask[kd_idx] = 1.0
            
            if kd_mask.sum() == 0:
                continue
            
            # Compute effective weight
            w_masked = weights * kd_mask.view(1, 1, kd, 1, 1)
            w_eff = w_masked.sum(dim=2)  # (out_c, in_c, kh, kw)
            
            # Create masked perturbation
            spatial_pert = spatial_mask.unsqueeze(0).unsqueeze(0).expand(B, in_c, -1, -1) * epsilon
            
            # Compute radius contribution via 2D convolution
            radius_2d = F.conv2d(
                spatial_pert,
                w_eff.abs(),
                bias=None,
                stride=(stride[1], stride[2]),
                padding=(padding[1], padding[2])
            )
            
            # Add to radius at this output depth
            radius[:, :, od, :, :] += radius_2d
    
    return radius


time: 7.43 ms (started: 2026-03-30 14:40:27 +01:00)


#### Closed form solution with Lipschitz bounds
In this method we evaluate the closed form solution by computing the effective weight on the first Convolution layer of the network followed by Lipschitz bounds on the remaining layers of the network

In [10]:
# =============================================================================
# Lipschitz-based Bound Propagation (L-2 norm space)
# =============================================================================
def compute_spectral_norm_conv3d(conv_layer: nn.Conv3d) -> float:
    """Compute spectral norm (L2 Lipschitz constant) of Conv3d layer."""
    weight = conv_layer.weight.detach()
    out_c, in_c, kd, kh, kw = weight.shape
    
    weight_2d = weight.reshape(out_c, -1)
    
    try:
        singular_values = torch.linalg.svdvals(weight_2d)
        return singular_values[0].item()
    except Exception:
        return torch.norm(weight_2d, p=2).item()


def compute_spectral_norm_linear(linear_layer: nn.Linear) -> float:
    """Compute spectral norm of Linear layer."""
    weight = linear_layer.weight.detach()
    
    try:
        singular_values = torch.linalg.svdvals(weight)
        return singular_values[0].item()
    except Exception:
        return torch.norm(weight, p=2).item()


def compute_spectral_norm_batchnorm(bn_layer: nn.BatchNorm3d) -> float:
    """
    Compute effective Lipschitz constant of BatchNorm layer.
    
    BN is essentially a diagonal linear transform:
    L = max_i |γ_i / √(σ²_i + ε)|
    """
    gamma = bn_layer.weight.detach()
    var = bn_layer.running_var.detach()
    eps = bn_layer.eps
    
    scale = gamma / torch.sqrt(var + eps)
    return scale.abs().max().item()


def compute_spectral_norm_adaptive_pool(
    input_shape: Tuple[int, int, int],
    output_shape: Tuple[int, int, int]
) -> float:
    """
    Compute Lipschitz constant of AdaptiveAvgPool3d.
    
    For average pooling with kernel size k, the Lipschitz constant is 1.
    (Each output is a weighted average of inputs with weights summing to 1)
    """
    return 1.0




def linf_to_l2_radius(linf_radius: torch.Tensor) -> torch.Tensor:
    """
    Convert L-infinity radius to L-2 radius.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    Given an L-infinity box: x ∈ [center - r, center + r] coordinate-wise,
    we want to find the L-2 radius R such that:
        ||x - center||_2 ≤ R  for all x in the box
    
    The maximum L-2 distance occurs at a corner of the box:
        max_{x in box} ||x - center||_2 = r × √n
    
    where n is the number of dimensions.
    
    However, if the radius varies per dimension (r_i), we have:
        max ||x - center||_2 = √(Σ_i r_i²)
    
    For uniform radius r: R = r × √n
    
    ============================================================================
    SOUNDNESS
    ============================================================================
    
    The L-2 ball with radius R = r × √n contains the entire L-infinity box.
    This is sound but potentially loose (especially for high-dimensional spaces).
    
    Args:
        linf_radius: L-infinity radius tensor (coordinate-wise)
        
    Returns:
        L-2 radius (scalar per sample in batch)
    """
    # Flatten to get all dimensions
    if linf_radius.dim() > 1:
        # Reshape to (batch, -1) to get all spatial/channel dimensions
        batch_size = linf_radius.shape[0]
        linf_flat = linf_radius.view(batch_size, -1)
        # Compute L-2 radius: sqrt(sum of squares of L-inf radii)
        # This is tighter than r * sqrt(n) when radii vary
        l2_radius = torch.norm(linf_flat, p=2, dim=1)  # Shape: (batch_size,)
    else:
        # Single sample, no batch dimension
        linf_flat = linf_radius.view(-1)
        l2_radius = torch.norm(linf_flat, p=2)  # Scalar tensor
    
    return l2_radius


def l2_radius_to_linf_bounds(center: torch.Tensor, l2_radius: torch.Tensor) -> LayerBounds:
    """
    Convert L-2 radius back to L-infinity bounds.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    Given center c and L-2 radius R, we want L-infinity bounds [ℓ, u] such that:
        For all x with ||x - c||_2 ≤ R: ℓ ≤ x ≤ u (coordinate-wise)
    
    The tightest L-infinity box containing the L-2 ball is:
        ℓ_i = c_i - R / √n
        u_i = c_i + R / √n
    
    where n is the number of dimensions.
    
    However, this is conservative. A tighter bound uses the fact that:
        max_{||x-c||_2 ≤ R} |x_i - c_i| = R  (achieved when x = c ± R·e_i)
    
    But since we want coordinate-wise bounds that hold simultaneously:
        ℓ_i = c_i - R
        u_i = c_i + R
    
    This is sound but loose. For tighter bounds, we could use the fact that
    the L-infinity norm of a vector in an L-2 ball is bounded by R.
    
    However, for verification purposes, we use the simpler bound:
        radius_linf = R / √n  (per dimension)
    
    This ensures that the L-infinity box contains the L-2 ball.
    
    Args:
        center: Center tensor (can have batch dimension)
        l2_radius: L-2 radius (scalar or tensor with batch dimension)
        
    Returns:
        LayerBounds with L-infinity bounds
    """
    # Ensure l2_radius is a tensor
    if not isinstance(l2_radius, torch.Tensor):
        l2_radius = torch.tensor(l2_radius, dtype=center.dtype, device=center.device)
    else:
        l2_radius = l2_radius.to(center.device)
    
    # Get number of dimensions (excluding batch)
    has_batch = center.dim() > 1
    if has_batch:
        batch_size = center.shape[0]
        n_dims = center.numel() // batch_size
        
        # Ensure l2_radius has batch dimension
        if l2_radius.dim() == 0:
            l2_radius = l2_radius.unsqueeze(0)
        if l2_radius.shape[0] == 1 and batch_size > 1:
            l2_radius = l2_radius.expand(batch_size)
        elif l2_radius.shape[0] != batch_size:
            raise ValueError(f"l2_radius batch size {l2_radius.shape[0]} doesn't match center batch size {batch_size}")
    else:
        # No batch dimension - ensure l2_radius is scalar
        n_dims = center.numel()
        if l2_radius.dim() > 0:
            if l2_radius.numel() > 1:
                raise ValueError("l2_radius has multiple values but center has no batch dimension")
            l2_radius = l2_radius.squeeze()
    
    # Convert to L-infinity radius per dimension
    # Using the bound: ||x||_∞ ≤ ||x||_2 / √n
    sqrt_n_dims = torch.sqrt(torch.tensor(n_dims, dtype=center.dtype, device=center.device))
    linf_radius_per_dim = l2_radius / sqrt_n_dims
    
    # Expand to match center shape
    if has_batch:
        linf_radius = linf_radius_per_dim.view(batch_size, *([1] * (center.dim() - 1)))
    else:
        linf_radius = linf_radius_per_dim
    
    lower = center - linf_radius
    upper = center + linf_radius
    
    return LayerBounds(lower=lower, upper=upper)


def l2_to_linf_radius(l2_radius: torch.Tensor, n_dims: int, device: str) -> torch.Tensor:
    """Convert L-2 radius to L-infinity radius per dimension."""
    sqrt_n = torch.sqrt(torch.tensor(n_dims, dtype=torch.float32, device=device))
    return l2_radius / sqrt_n



def lipschitz_propagate_cnn3d(
    first_layer_bounds: LayerBounds,
    model: CNN3D,
    device: str = DEVICE
) -> Tuple[LayerBounds, float]:
    """
    Propagate bounds through CNN3D using Lipschitz constants.
    
    Converts L∞ bounds to L2 radius, propagates via Lipschitz, converts back.
    
    Args:
        first_layer_bounds: Bounds from first layer
        model: CNN3D model
        device: Computation device
        
    Returns:
        (final_bounds, cumulative_lipschitz_constant)
    """
    # Ensure batch dimension
    lower = first_layer_bounds.lower
    upper = first_layer_bounds.upper
    
    if lower.dim() == 4:
        lower = lower.unsqueeze(0)
        upper = upper.unsqueeze(0)
    
    lower = lower.to(device)
    upper = upper.to(device)
    
    # Center-radius representation
    center = (lower + upper) / 2
    radius_linf = (upper - lower) / 2
    
    # Convert to L2 radius
    l2_radius = linf_to_l2_radius(radius_linf)
    
    cumulative_lipschitz = 1.0
    
    # BN1 Lipschitz constant
    lip_bn1 = compute_spectral_norm_batchnorm(model.bn1)
    l2_radius = l2_radius * lip_bn1
    cumulative_lipschitz *= lip_bn1
    
    # Apply ReLU (1-Lipschitz)
    # Need to convert to bounds, apply ReLU, convert back
    n_dims = center.numel() // center.shape[0]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)
    
    lower_tmp = center - linf_r.view(-1, 1, 1, 1, 1)
    upper_tmp = center + linf_r.view(-1, 1, 1, 1, 1)
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius_linf = (upper_out - lower_out) / 2
    l2_radius = linf_to_l2_radius(radius_linf)
    
    # Conv2 + BN2 (get folded weights for center computation)
    w2_folded, b2_folded = get_folded_conv_params(model.conv2, model.bn2, device)
    center = F.conv3d(center, w2_folded, b2_folded,
                      stride=model.s2, padding=model.pd2)
    
    lip_conv2 = compute_spectral_norm_conv3d(model.conv2)
    lip_bn2 = compute_spectral_norm_batchnorm(model.bn2)
    l2_radius = l2_radius * lip_conv2 * lip_bn2
    cumulative_lipschitz *= lip_conv2 * lip_bn2
    
    # ReLU
    n_dims = center.numel() // center.shape[0]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)
    
    lower_tmp = center - linf_r.view(-1, 1, 1, 1, 1)
    upper_tmp = center + linf_r.view(-1, 1, 1, 1, 1)
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius_linf = (upper_out - lower_out) / 2
    l2_radius = linf_to_l2_radius(radius_linf)
    
    # Adaptive Average Pooling (1-Lipschitz)
    center = model.adaptive_pool(center)
    # L2 radius unchanged by avg pool
    
    # Flatten
    center = center.view(center.size(0), -1)
    
    # FC1
    center = F.linear(center, model.fc1.weight, model.fc1.bias)
    lip_fc1 = compute_spectral_norm_linear(model.fc1)
    l2_radius = l2_radius * lip_fc1
    cumulative_lipschitz *= lip_fc1
    
    # ReLU
    n_dims = center.shape[1]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)
    
    lower_tmp = center - linf_r.unsqueeze(1)
    upper_tmp = center + linf_r.unsqueeze(1)
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius_linf = (upper_out - lower_out) / 2
    l2_radius = linf_to_l2_radius(radius_linf)
    
    # FC2
    center = F.linear(center, model.fc2.weight, model.fc2.bias)
    lip_fc2 = compute_spectral_norm_linear(model.fc2)
    l2_radius = l2_radius * lip_fc2
    cumulative_lipschitz *= lip_fc2
    
    # ReLU
    n_dims = center.shape[1]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)
    
    lower_tmp = center - linf_r.unsqueeze(1)
    upper_tmp = center + linf_r.unsqueeze(1)
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius_linf = (upper_out - lower_out) / 2
    l2_radius = linf_to_l2_radius(radius_linf)
    
    # FC3 (final layer)
    center = F.linear(center, model.fc3.weight, model.fc3.bias)
    lip_fc3 = compute_spectral_norm_linear(model.fc3)
    l2_radius = l2_radius * lip_fc3
    cumulative_lipschitz *= lip_fc3
    
    # Convert back to L∞ bounds
    n_dims = center.shape[1]
    linf_r = l2_to_linf_radius(l2_radius, n_dims, device)
    
    final_lower = center - linf_r.unsqueeze(1)
    final_upper = center + linf_r.unsqueeze(1)
    
    return LayerBounds(lower=final_lower, upper=final_upper), cumulative_lipschitz


time: 8.35 ms (started: 2026-03-30 14:40:38 +01:00)


In [11]:
def closed_form_lipschitz_verify_cnn3d(
    model: CNN3D,
    input_tensor: torch.Tensor,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    device: str = DEVICE
) -> Tuple[torch.Tensor, torch.Tensor, float, float]:
    """
    Complete verification: Closed-form first layer + Lipschitz propagation.
    
    Args:
        model: CNN3D model
        input_tensor: Input tensor
        epsilon: Perturbation budget
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask
        device: Computation device
        
    Returns:
        (lower_bounds, upper_bounds, runtime_seconds, lipschitz_constant)
    """
    start_time = time.time()
    
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    model = model.to(device)
    model.eval()
    input_tensor = input_tensor.to(device)
    
    # Step 1: Closed-form bounds on first Conv3D + BN1
    first_layer_bounds = closed_form_conv3d_bounds(
        input_tensor,
        model.conv1,
        epsilon,
        shared_frames,
        spatial_mask,
        model.bn1,
        device
    )
    
    # Step 2: Lipschitz propagation
    final_bounds, lipschitz_const = lipschitz_propagate_cnn3d(
        first_layer_bounds,
        model,
        device
    )
    
    runtime = time.time() - start_time
    
    return final_bounds.lower, final_bounds.upper, runtime, lipschitz_const


time: 1.61 ms (started: 2026-03-30 14:40:45 +01:00)


In [19]:
closed_form_lips_rob_acc = []
clean_acc = []

epsilons = [3, 2.75, 2.50, 2.25, 2, 1.75, 1.5, 1.25, 1, 0.75, 0.5, 0.25, 0.1, 0.075, 0.05, 0.025, 0.01, 0.0075, 0.005, 0.0025, 0.001, 0.0001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4,5,6,7,8,9]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.to(DEVICE).to(torch.long)

        final_lower, final_upper, _, _ = closed_form_lipschitz_verify_cnn3d(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_lips_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-Lipschitz Robust Acc: {closed_form_lips_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-Lipschitz Robust Accuracy: ", closed_form_lips_rob_acc)

# print(correct," ",robust, " ",total)

Epsilon: 3 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 2.75 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 2.5 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 2.25 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 0.00%


KeyboardInterrupt: 

time: 11min 3s (started: 2026-01-20 03:07:59 +00:00)


In [20]:
closed_form_lips_rob_acc = []
clean_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.to(DEVICE).to(torch.long)
                        
        final_lower, final_upper, _, _ = closed_form_lipschitz_verify_cnn3d(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_lips_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-Lipschitz Robust Acc: {closed_form_lips_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-Lipschitz Robust Accuracy: ", closed_form_lips_rob_acc)

# print(correct," ",robust, " ",total)

Epsilon: 0.1 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 0.01 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 0.00%
Epsilon: 0.001 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 74.27%
Epsilon: 0.0001 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 84.24%
Epsilon: 1e-05 => Clean Acc: 84.24%, Closed-form-Lipschitz Robust Acc: 84.24%
Model Accuracy:  [84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579]
Closed-form-Lipschitz Robust Accuracy:  [0.0, 0.0, 74.2653606411398, 84.23864648263579, 84.23864648263579]
time: 10min 1s (started: 2026-01-20 03:23:07 +00:00)


#### Closed form solution with IBP bounds
In this method we evaluate the closed form solution by computing the effective weight on the first Convolution layer of the network followed by IBP bounds on the remaining layers of the network

In [25]:
def ibp_propagate_cnn3d(
    first_layer_bounds: LayerBounds,
    model: CNN3D,
    device: str = DEVICE
) -> Tuple[LayerBounds, float]:
    """
    Propagate bounds through CNN3D using Interval Bound Propagation.
    
    Handles: Conv3D, BatchNorm3D, ReLU, AdaptiveAvgPool3d, Linear
    
    Args:
        first_layer_bounds: Bounds from first layer (pre-BN, pre-ReLU)
        model: CNN3D model
        device: Computation device
        
    Returns:
        (final_bounds, cumulative_spectral_norm)
    """
    # Ensure batch dimension
    lower = first_layer_bounds.lower
    upper = first_layer_bounds.upper
    
    if lower.dim() == 4:
        lower = lower.unsqueeze(0)
        upper = upper.unsqueeze(0)
    
    lower = lower.to(device)
    upper = upper.to(device)
    
    # Center-radius representation
    center = (lower + upper) / 2
    radius = (upper - lower) / 2
    
    cumulative_spectral_norm = 1.0
    
    # Note: First layer bounds already include conv1, so we start from bn1
    # Apply BN1 (folded into conv1 in closed-form, but need to track for spectral norm)
    cumulative_spectral_norm *= compute_spectral_norm_batchnorm(model.bn1)
    
    # Apply ReLU
    lower_tmp = center - radius
    upper_tmp = center + radius
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius = (upper_out - lower_out) / 2
    
    # Conv2 + BN2
    # Get folded weights
    w2_folded, b2_folded = get_folded_conv_params(model.conv2, model.bn2, device)
    
    center = F.conv3d(center, w2_folded, b2_folded, 
                      stride=model.s2, padding=model.pd2)
    radius = F.conv3d(radius, w2_folded.abs(), None,
                      stride=model.s2, padding=model.pd2)
    
    cumulative_spectral_norm *= compute_spectral_norm_conv3d(model.conv2)
    cumulative_spectral_norm *= compute_spectral_norm_batchnorm(model.bn2)
    
    # Apply ReLU
    lower_tmp = center - radius
    upper_tmp = center + radius
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius = (upper_out - lower_out) / 2
    
    # Adaptive Average Pooling
    # For IBP, we can propagate through adaptive avg pool
    center = model.adaptive_pool(center)
    radius = model.adaptive_pool(radius)  # Average preserves IBP bounds
    
    cumulative_spectral_norm *= 1.0  # Avg pool is 1-Lipschitz
    
    # Flatten
    center = center.view(center.size(0), -1)
    radius = radius.view(radius.size(0), -1)
    
    # FC1
    center = F.linear(center, model.fc1.weight, model.fc1.bias)
    radius = F.linear(radius, model.fc1.weight.abs(), None)
    cumulative_spectral_norm *= compute_spectral_norm_linear(model.fc1)
    
    # ReLU
    lower_tmp = center - radius
    upper_tmp = center + radius
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius = (upper_out - lower_out) / 2
    
    # FC2
    center = F.linear(center, model.fc2.weight, model.fc2.bias)
    radius = F.linear(radius, model.fc2.weight.abs(), None)
    cumulative_spectral_norm *= compute_spectral_norm_linear(model.fc2)
    
    # ReLU
    lower_tmp = center - radius
    upper_tmp = center + radius
    lower_out = F.relu(lower_tmp)
    upper_out = F.relu(upper_tmp)
    center = (lower_out + upper_out) / 2
    radius = (upper_out - lower_out) / 2
    
    # FC3 (final layer, no ReLU)
    center = F.linear(center, model.fc3.weight, model.fc3.bias)
    radius = F.linear(radius, model.fc3.weight.abs(), None)
    cumulative_spectral_norm *= compute_spectral_norm_linear(model.fc3)
    
    # Final bounds
    final_lower = center - radius
    final_upper = center + radius
    
    return LayerBounds(lower=final_lower, upper=final_upper), cumulative_spectral_norm



time: 2.8 ms (started: 2026-01-20 03:45:02 +00:00)


In [22]:
def closed_form_ibp_verify_cnn3d(
    model: CNN3D,
    input_tensor: torch.Tensor,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    device: str = DEVICE
) -> Tuple[torch.Tensor, torch.Tensor, float, float]:
    """
    Complete verification: Closed-form first layer + IBP propagation.
    
    Args:
        model: CNN3D model
        input_tensor: Input tensor (C, D, H, W) or (B, C, D, H, W)
        epsilon: Perturbation budget
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask for patch attacks
        device: Computation device
        
    Returns:
        (lower_bounds, upper_bounds, runtime_seconds, spectral_norm)
    """
    start_time = time.time()
    
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    model = model.to(device)
    model.eval()
    input_tensor = input_tensor.to(device)
    
    # Step 1: Closed-form bounds on first Conv3D + BN1
    first_layer_bounds = closed_form_conv3d_bounds(
        input_tensor,
        model.conv1,
        epsilon,
        shared_frames,
        spatial_mask,
        model.bn1,  # Fold BatchNorm into Conv
        device
    )
    
    # Step 2: IBP propagation
    final_bounds, spectral_norm = ibp_propagate_cnn3d(
        first_layer_bounds,
        model,
        device
    )
    
    runtime = time.time() - start_time
    
    return final_bounds.lower, final_bounds.upper, runtime, spectral_norm



time: 1.5 ms (started: 2026-01-20 03:43:34 +00:00)


In [ ]:
closed_form_ibp_rob_acc = []
clean_acc = []

epsilons = [3, 2.75, 2.50, 2.25, 2, 1.75, 1.5, 1.25, 1, 0.75, 0.5, 0.25, 0.1, 0.075, 0.05, 0.025, 0.01, 0.0075, 0.005, 0.0025, 0.001, 0.0001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4,5,6,7,8,9]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.squeeze(1).to(DEVICE).to(torch.long)
                        
        final_lower, final_upper, _, _ = closed_form_ibp_verify(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_ibp_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-IBP Robust Acc: {closed_form_ibp_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-IBP Robust Accuracy: ", closed_form_ibp_rob_acc)
# print(correct," ",robust, " ",total)

Epsilon: 3 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 2.75 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 2.5 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 2.25 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 2 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 1.75 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 1.5 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 1.25 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 1 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.75 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.5 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.25 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.1 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.075 => Clean Acc: 72.88%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.05 => Clean Acc: 72.8

In [26]:
closed_form_ibp_rob_acc = []
clean_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4,5,6,7,8,9]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.to(DEVICE).to(torch.long)
                        
        final_lower, final_upper, _, _ = closed_form_ibp_verify_cnn3d(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_ibp_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-IBP Robust Acc: {closed_form_ibp_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-IBP Robust Accuracy: ", closed_form_ibp_rob_acc)
# print(correct," ",robust, " ",total)

Epsilon: 0.1 => Clean Acc: 84.24%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.01 => Clean Acc: 84.24%, Closed-form-IBP Robust Acc: 0.00%
Epsilon: 0.001 => Clean Acc: 84.24%, Closed-form-IBP Robust Acc: 68.12%
Epsilon: 0.0001 => Clean Acc: 84.24%, Closed-form-IBP Robust Acc: 84.24%
Epsilon: 1e-05 => Clean Acc: 84.24%, Closed-form-IBP Robust Acc: 84.24%
Model Accuracy:  [84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579]
Closed-form-IBP Robust Accuracy:  [0.0, 0.0, 68.12110418521816, 84.23864648263579, 84.23864648263579]
time: 9min 27s (started: 2026-01-20 03:45:09 +00:00)


#### Closed form solution with Löwner John Ellipsoid Sampling
In this method we evaluate the closed form solution by computing the effective weight on the first Convolution layer of the network followed by Löwner John Ellipsoid Sampling on the remaining layers of the network

In [ ]:
# =============================================================================
# Löwner-John Ellipsoid Propagation for CNN3D
# =============================================================================

def linf_bounds_to_ellipsoid(
    bounds: LayerBounds,
    device: str = DEVICE,
    max_dims: int = 500
) -> Ellipsoid:
    """
    Convert L∞ bounds to approximating Löwner-John ellipsoid.
    
    For box with radius r_i per dimension:
        P = diag(n / r_i²)
    """
    center = bounds.center.to(device)
    radius = bounds.radius.to(device)
    
    if center.dim() > 1:
        batch_size = center.shape[0]
        center_flat = center.view(batch_size, -1)
        radius_flat = radius.view(batch_size, -1)
        n_dims = center_flat.shape[1]
        
        r_sq = radius_flat ** 2
        r_sq = torch.clamp(r_sq, min=1e-8)
        P_diag = n_dims / r_sq
        
        return Ellipsoid(center=center, P=P_diag, is_diagonal=True)
    else:
        center_flat = center.view(-1)
        radius_flat = radius.view(-1)
        n_dims = center_flat.shape[0]
        
        r_sq = radius_flat ** 2
        r_sq = torch.clamp(r_sq, min=1e-8)
        P_diag = n_dims / r_sq
        
        return Ellipsoid(center=center, P=P_diag, is_diagonal=True)


def sample_ellipsoid_bounds(
    ellipsoid: Ellipsoid,
    n_samples: int = 1000,
    chunk_size: int = 100,
    device: str = DEVICE
) -> LayerBounds:
    """
    Sample points on ellipsoid surface to estimate bounds.
    
    For P = diag(p_i), sampling: x_i = c_i + u_i / √p_i
    where u is on unit sphere.
    """
    center = ellipsoid.center
    P_diag = ellipsoid.P
    
    has_batch = P_diag.dim() > 1
    
    if has_batch:
        batch_size = P_diag.shape[0]
        n_dims = P_diag.shape[1]
        center_flat = center.view(batch_size, -1)
        
        lower = torch.full((batch_size, n_dims), float('inf'), device=device)
        upper = torch.full((batch_size, n_dims), float('-inf'), device=device)
        
        scale = 1.0 / torch.sqrt(torch.clamp(P_diag, min=1e-8))
        
        for b_idx in range(batch_size):
            center_b = center_flat[b_idx]
            scale_b = scale[b_idx]
            
            lower_b = torch.full((n_dims,), float('inf'), device=device)
            upper_b = torch.full((n_dims,), float('-inf'), device=device)
            
            n_chunks = (n_samples + chunk_size - 1) // chunk_size
            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * chunk_size
                end_idx = min(start_idx + chunk_size, n_samples)
                chunk_n = end_idx - start_idx
                
                u_chunk = torch.randn(chunk_n, n_dims, device=device)
                u_chunk = u_chunk / torch.norm(u_chunk, p=2, dim=1, keepdim=True)
                
                samples_chunk = center_b.unsqueeze(0) + u_chunk * scale_b.unsqueeze(0)
                
                lower_b = torch.minimum(lower_b, samples_chunk.min(dim=0).values)
                upper_b = torch.maximum(upper_b, samples_chunk.max(dim=0).values)
                
                del u_chunk, samples_chunk
            
            lower[b_idx] = lower_b
            upper[b_idx] = upper_b
        
        center_shape = center.shape
        lower = lower.view(center_shape)
        upper = upper.view(center_shape)
    else:
        n_dims = P_diag.shape[0]
        center_flat = center.view(-1)
        
        scale = 1.0 / torch.sqrt(torch.clamp(P_diag, min=1e-8))
        
        lower_flat = torch.full((n_dims,), float('inf'), device=device)
        upper_flat = torch.full((n_dims,), float('-inf'), device=device)
        
        n_chunks = (n_samples + chunk_size - 1) // chunk_size
        for chunk_idx in range(n_chunks):
            start_idx = chunk_idx * chunk_size
            end_idx = min(start_idx + chunk_size, n_samples)
            chunk_n = end_idx - start_idx
            
            u_chunk = torch.randn(chunk_n, n_dims, device=device)
            u_chunk = u_chunk / torch.norm(u_chunk, p=2, dim=1, keepdim=True)
            
            samples_chunk = center_flat.unsqueeze(0) + u_chunk * scale.unsqueeze(0)
            
            lower_flat = torch.minimum(lower_flat, samples_chunk.min(dim=0).values)
            upper_flat = torch.maximum(upper_flat, samples_chunk.max(dim=0).values)
            
            del u_chunk, samples_chunk
        
        lower = lower_flat.view(center.shape)
        upper = upper_flat.view(center.shape)
    
    return LayerBounds(lower=lower, upper=upper)


def propagate_ellipsoid_affine(
    ellipsoid: Ellipsoid,
    W: torch.Tensor,
    b: Optional[torch.Tensor] = None,
    device: str = DEVICE
) -> Ellipsoid:
    """
    Propagate ellipsoid through affine transformation y = Wx + b.
    
    For diagonal P_in = diag(p_i):
        P_out = (W @ diag(1/p_i) @ W^T)^{-1}
    """
    center_in = ellipsoid.center
    P_in_diag = ellipsoid.P
    
    has_batch = P_in_diag.dim() > 1
    
    if has_batch:
        batch_size = P_in_diag.shape[0]
        if center_in.dim() > 2:
            center_in_flat = center_in.view(batch_size, -1)
        else:
            center_in_flat = center_in
        
        center_out_flat = F.linear(center_in_flat, W, bias=b)
        n_out = center_out_flat.shape[1]
        
        P_inv_diag = 1.0 / torch.clamp(P_in_diag, min=1e-8)
        
        P_out_list = []
        for b_idx in range(batch_size):
            P_inv_b = P_inv_diag[b_idx]
            W_scaled = W * P_inv_b.unsqueeze(0)
            W_Pinv_WT = W_scaled @ W.T
            
            # Use diagonal approximation for memory efficiency
            P_out_diag = 1.0 / torch.clamp(torch.diag(W_Pinv_WT), min=1e-8)
            P_out_list.append(P_out_diag)
        
        P_out = torch.stack(P_out_list)
        return Ellipsoid(center=center_out_flat, P=P_out, is_diagonal=True)
    else:
        if center_in.dim() > 1:
            center_in_flat = center_in.view(-1)
        else:
            center_in_flat = center_in
        
        center_out_flat = F.linear(center_in_flat.unsqueeze(0), W, bias=b).squeeze(0)
        
        P_inv_diag = 1.0 / torch.clamp(P_in_diag, min=1e-8)
        W_scaled = W * P_inv_diag.unsqueeze(0)
        W_Pinv_WT = W_scaled @ W.T
        
        P_out_diag = 1.0 / torch.clamp(torch.diag(W_Pinv_WT), min=1e-8)
        
        return Ellipsoid(center=center_out_flat, P=P_out_diag, is_diagonal=True)


def loewner_john_propagate_cnn3d(
    first_layer_bounds: LayerBounds,
    model: CNN3D,
    n_samples: int = 500,
    chunk_size: int = 100,
    device: str = DEVICE
) -> Tuple[LayerBounds, int]:
    """
    Propagate bounds through CNN3D using Löwner-John ellipsoid sampling.
    
    Args:
        first_layer_bounds: Bounds from first layer
        model: CNN3D model
        n_samples: Number of samples for ellipsoid
        chunk_size: Chunk size for memory efficiency
        device: Computation device
        
    Returns:
        (final_bounds, n_samples_used)
    """
    # Ensure batch dimension
    lower = first_layer_bounds.lower
    upper = first_layer_bounds.upper
    
    if lower.dim() == 4:
        lower = lower.unsqueeze(0)
        upper = upper.unsqueeze(0)
    
    lower = lower.to(device)
    upper = upper.to(device)
    
    # Convert to ellipsoid
    bounds = LayerBounds(lower=lower, upper=upper)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # Apply BN1 + ReLU (convert to bounds, apply, convert back)
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # Conv2 + BN2 (use IBP-style for conv, then re-ellipsoidize)
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    center = (temp_bounds.lower + temp_bounds.upper) / 2
    radius = (temp_bounds.upper - temp_bounds.lower) / 2
    
    w2_folded, b2_folded = get_folded_conv_params(model.conv2, model.bn2, device)
    
    center = F.conv3d(center, w2_folded, b2_folded,
                      stride=model.s2, padding=model.pd2)
    radius = F.conv3d(radius, w2_folded.abs(), None,
                      stride=model.s2, padding=model.pd2)
    
    lower_out = center - radius
    upper_out = center + radius
    bounds = LayerBounds(lower=lower_out, upper=upper_out)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # ReLU
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # Adaptive Average Pooling
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_pool = model.adaptive_pool(temp_bounds.lower)
    upper_pool = model.adaptive_pool(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_pool, upper=upper_pool)
    
    # Flatten
    bounds = LayerBounds(
        lower=bounds.lower.view(bounds.lower.size(0), -1),
        upper=bounds.upper.view(bounds.upper.size(0), -1)
    )
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # FC1
    ellipsoid = propagate_ellipsoid_affine(ellipsoid, model.fc1.weight, model.fc1.bias, device)
    
    # ReLU
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # FC2
    ellipsoid = propagate_ellipsoid_affine(ellipsoid, model.fc2.weight, model.fc2.bias, device)
    
    # ReLU
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # FC3
    ellipsoid = propagate_ellipsoid_affine(ellipsoid, model.fc3.weight, model.fc3.bias, device)
    
    # Final sampling
    final_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    
    return final_bounds, n_samples

NameError: name 'Simple3DCNN' is not defined

time: 198 ms (started: 2026-01-20 03:59:51 +00:00)


In [ ]:
closed_form_ljsamp_rob_acc = []
clean_acc = []

epsilons = [3, 2.75, 2.50, 2.25, 2, 1.75, 1.5, 1.25, 1, 0.75, 0.5, 0.25, 0.1, 0.075, 0.05, 0.025, 0.01, 0.0075, 0.005, 0.0025, 0.001, 0.0001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4,5,6,7,8,9]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.squeeze(1).to(DEVICE).to(torch.long)
                        
        final_lower, final_upper, _, _ = closed_form_lowner_john_verify(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE, n_samples=10, chunk_size=10
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_ljsamp_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-Löwner-John Sampling Robust Acc: {closed_form_ljsamp_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-Löwner-John Sampling Robust Accuracy: ", closed_form_ljsamp_rob_acc)

# print(correct," ",robust, " ",total)

Epsilon: 3 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 2.75 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 2.5 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 2.25 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 2 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 1.75 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 1.5 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 1.25 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 1 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 0.75 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 0.5 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 0.25 => Clean Acc: 72.88%, Closed-f

In [ ]:
closed_form_ljsamp_rob_acc = []
clean_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.squeeze(1).to(DEVICE).to(torch.long)
                        
        final_lower, final_upper, _, _ = closed_form_lowner_john_verify(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE, n_samples=10, chunk_size=10
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_ljsamp_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-Löwner-John Sampling Robust Acc: {closed_form_ljsamp_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-Löwner-John Sampling Robust Accuracy: ", closed_form_ljsamp_rob_acc)

# print(correct," ",robust, " ",total)

Epsilon: 0.1 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 0.01 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 0.001 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 0.0001 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Epsilon: 1e-05 => Clean Acc: 72.88%, Closed-form-Löwner-John Sampling Robust Acc: 72.88%
Model Accuracy:  [72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339]
Closed-form-Löwner-John Sampling Robust Accuracy:  [72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339]
time: 13.7 s (started: 2026-01-19 21:53:50 +00:00)


#### Selective Interval Bound Propagation - Adversarial Patches

For the adversarial patches, we implement the closed form solution with Lipschitz bounds propagation.

In [50]:

def closed_form_conv3d_bounds_patch(
    input_tensor: torch.Tensor,
    conv_layer: nn.Conv3d,
    epsilon: float,
    frame_patch_locations: Dict[int, List[Tuple[int, int, int, int]]],
    device: str = DEVICE
) -> LayerBounds:
    """
    Compute EXACT bounds on first Conv3D layer output for adversarial patch constraints.
    
    ============================================================================
    MATHEMATICAL DERIVATION
    ============================================================================
    
    For adversarial patch perturbations, the perturbation model is:
        x_pert[c,d,h,w] = x[c,d,h,w] + δ[c,h,w] × mask[d,h,w]
    
    Where:
        - δ[c,h,w] ∈ [-ε, ε] is the shared spatial perturbation
        - mask[d,h,w] = 1 if (h,w) is inside a patch region for frame d, else 0
    
    The perturbation contribution to output is:
        Δz[o,od,oh,ow] = Σ_{c,kd,kh,kw} W[o,c,kd,kh,kw] × δ[c,ih,iw] × mask[id,ih,iw]
    
    Where:
        - id = od×stride_d + kd - padding_d
        - ih = oh×stride_h + kh - padding_h
        - iw = ow×stride_w + kw - padding_w
    
    For the optimization problem:
        max/min  Σ W × δ × mask
        s.t.     δ ∈ [-ε, ε]^n
    
    The optimal solution is:
        δ*[c,h,w] = ε × sign(Σ_{kd: mask[id,h,w]=1} W[o,c,kd,kh,kw])  for max
    
    Therefore:
        radius[o,od,oh,ow] = ε × Σ_{c,kd,kh,kw} |W[o,c,kd,kh,kw]| × mask[id,ih,iw]
    
    This is equivalent to: conv3d(mask × ε, |W|)
    
    ============================================================================
    SOUNDNESS PROOF
    ============================================================================
    
    Claim: For all valid δ ∈ [-ε,ε] with patch constraints: lower ≤ Conv3D(x + δ×mask) ≤ upper
    
    Proof:
    1. The perturbation contribution is: Δz = Σ W × δ × mask
    2. For any δ ∈ [-ε, ε]:
       |W × δ × mask| ≤ |W| × ε × mask
    3. Summing: |Δz| ≤ ε × Σ |W| × mask = conv3d(mask × ε, |W|)
    4. Therefore: z_clean - radius ≤ z_pert ≤ z_clean + radius
    
    QED. The bounds are sound and tight (achieved when δ = ε × sign(W) × mask).
    
    ============================================================================
    
    Args:
        input_tensor: Input tensor of shape (C, D, H, W) or (B, C, D, H, W)
        conv_layer: First Conv3d layer
        epsilon: Perturbation budget (L∞ norm)
        frame_patch_locations: Dictionary mapping frame index to list of patch bounding boxes
                               Each box is (x1, y1, x2, y2) in pixel coordinates
                               where x corresponds to width and y to height
        device: Computation device
        
    Returns:
        LayerBounds containing lower and upper bounds on Conv3D output
        
    Complexity:
        O(B × forward_pass) - same as standard convolution, fully GPU-parallelized
    """
    # Handle input dimensions
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    B, C, D, H, W = input_tensor.shape
    input_tensor = input_tensor.to(device)
    
    # Get layer parameters
    weights = conv_layer.weight.to(device)  # (out_c, in_c, kd, kh, kw)
    bias = conv_layer.bias.to(device) if conv_layer.bias is not None else None
    
    out_c, in_c, kd, kh, kw = weights.shape
    
    # Handle stride and padding (can be int or tuple)
    if isinstance(conv_layer.stride, int):
        stride = (conv_layer.stride, conv_layer.stride, conv_layer.stride)
    else:
        stride = conv_layer.stride
        
    if isinstance(conv_layer.padding, int):
        padding = (conv_layer.padding, conv_layer.padding, conv_layer.padding)
    else:
        padding = conv_layer.padding
    
    # Compute clean output: z(x)
    z_clean = F.conv3d(input_tensor, weights, bias, stride=stride, padding=padding)
    
    # Get output dimensions from clean forward pass
    out_D = z_clean.shape[2]
    out_H = z_clean.shape[3]
    out_W = z_clean.shape[4]
    
    # =========================================================================
    # Create patch mask for each frame
    # mask[d, h, w] = 1 if pixel (h, w) is inside any patch in frame d
    # =========================================================================
    mask = torch.zeros(D, H, W, device=device)
    min_frame_img_num = min(frame_patch_locations.keys())
    for frame_img_num, box in frame_patch_locations.items():
        frame_idx = frame_img_num - min_frame_img_num  # Map to 0-based index
        if 0 <= frame_idx < D:
            (x1, y1, x2, y2) = box[0]  # Unpack bounding box
            # Clamp to valid indices
            y1_clamped = max(0, min(int(y1), H))
            y2_clamped = max(0, min(int(y2), H))
            x1_clamped = max(0, min(int(x1), W))
            x2_clamped = max(0, min(int(x2), W))
        
        # Set mask to 1 inside patch region
        # Note: y corresponds to height dimension, x to width
        mask[frame_idx, y1_clamped:y2_clamped, x1_clamped:x2_clamped] = 1.0
    
    # =========================================================================
    # Compute radius via 3D convolution with absolute weights
    # =========================================================================
    # 
    # Key insight: The radius for patch perturbations is:
    #   radius[o,od,oh,ow] = ε × Σ_{c,kd,kh,kw} |W[o,c,kd,kh,kw]| × mask[id,ih,iw]
    #
    # This is exactly: conv3d(mask × ε, |W|)
    #
    # Expand mask to have input channel dimension: (1, in_c, D, H, W)
    # The mask is the same for all input channels (perturbation is per spatial location)
    
    # Create perturbation indicator tensor with epsilon magnitude
    # Shape: (1, in_c, D, H, W)
    mask_5d = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, D, H, W)
    mask_expanded = mask_5d.expand(1, in_c, D, H, W) * epsilon  # (1, in_c, D, H, W)
    
    # Compute radius via 3D convolution with absolute weights
    # This computes for each output position:
    #   Σ_{c,kd,kh,kw} |W[o,c,kd,kh,kw]| × (mask × ε)[id,ih,iw]
    W_abs = weights.abs()  # (out_c, in_c, kd, kh, kw)
    
    radius = F.conv3d(
        mask_expanded,
        W_abs,
        bias=None,
        stride=stride,
        padding=padding
    )  # Shape: (1, out_c, out_D, out_H, out_W)
    
    # Expand radius to batch size
    radius = radius.expand(B, -1, -1, -1, -1)
    
    # Compute final bounds
    lower_bounds = z_clean - radius
    upper_bounds = z_clean + radius
    
    return LayerBounds(lower=lower_bounds, upper=upper_bounds)



time: 8.34 ms (started: 2026-01-20 05:12:08 +00:00)


In [32]:
# =============================================================================
# Löwner-John Ellipsoid Propagation for CNN3D
# =============================================================================

def linf_bounds_to_ellipsoid(
    bounds: LayerBounds,
    device: str = DEVICE,
    max_dims: int = 500
) -> Ellipsoid:
    """
    Convert L∞ bounds to approximating Löwner-John ellipsoid.
    
    For box with radius r_i per dimension:
        P = diag(n / r_i²)
    """
    center = bounds.center.to(device)
    radius = bounds.radius.to(device)
    
    if center.dim() > 1:
        batch_size = center.shape[0]
        center_flat = center.view(batch_size, -1)
        radius_flat = radius.view(batch_size, -1)
        n_dims = center_flat.shape[1]
        
        r_sq = radius_flat ** 2
        r_sq = torch.clamp(r_sq, min=1e-8)
        P_diag = n_dims / r_sq
        
        return Ellipsoid(center=center, P=P_diag, is_diagonal=True)
    else:
        center_flat = center.view(-1)
        radius_flat = radius.view(-1)
        n_dims = center_flat.shape[0]
        
        r_sq = radius_flat ** 2
        r_sq = torch.clamp(r_sq, min=1e-8)
        P_diag = n_dims / r_sq
        
        return Ellipsoid(center=center, P=P_diag, is_diagonal=True)


def sample_ellipsoid_bounds(
    ellipsoid: Ellipsoid,
    n_samples: int = 1000,
    chunk_size: int = 100,
    device: str = DEVICE
) -> LayerBounds:
    """
    Sample points on ellipsoid surface to estimate bounds.
    
    For P = diag(p_i), sampling: x_i = c_i + u_i / √p_i
    where u is on unit sphere.
    """
    center = ellipsoid.center
    P_diag = ellipsoid.P
    
    has_batch = P_diag.dim() > 1
    
    if has_batch:
        batch_size = P_diag.shape[0]
        n_dims = P_diag.shape[1]
        center_flat = center.view(batch_size, -1)
        
        lower = torch.full((batch_size, n_dims), float('inf'), device=device)
        upper = torch.full((batch_size, n_dims), float('-inf'), device=device)
        
        scale = 1.0 / torch.sqrt(torch.clamp(P_diag, min=1e-8))
        
        for b_idx in range(batch_size):
            center_b = center_flat[b_idx]
            scale_b = scale[b_idx]
            
            lower_b = torch.full((n_dims,), float('inf'), device=device)
            upper_b = torch.full((n_dims,), float('-inf'), device=device)
            
            n_chunks = (n_samples + chunk_size - 1) // chunk_size
            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * chunk_size
                end_idx = min(start_idx + chunk_size, n_samples)
                chunk_n = end_idx - start_idx
                
                u_chunk = torch.randn(chunk_n, n_dims, device=device)
                u_chunk = u_chunk / torch.norm(u_chunk, p=2, dim=1, keepdim=True)
                
                samples_chunk = center_b.unsqueeze(0) + u_chunk * scale_b.unsqueeze(0)
                
                lower_b = torch.minimum(lower_b, samples_chunk.min(dim=0).values)
                upper_b = torch.maximum(upper_b, samples_chunk.max(dim=0).values)
                
                del u_chunk, samples_chunk
            
            lower[b_idx] = lower_b
            upper[b_idx] = upper_b
        
        center_shape = center.shape
        lower = lower.view(center_shape)
        upper = upper.view(center_shape)
    else:
        n_dims = P_diag.shape[0]
        center_flat = center.view(-1)
        
        scale = 1.0 / torch.sqrt(torch.clamp(P_diag, min=1e-8))
        
        lower_flat = torch.full((n_dims,), float('inf'), device=device)
        upper_flat = torch.full((n_dims,), float('-inf'), device=device)
        
        n_chunks = (n_samples + chunk_size - 1) // chunk_size
        for chunk_idx in range(n_chunks):
            start_idx = chunk_idx * chunk_size
            end_idx = min(start_idx + chunk_size, n_samples)
            chunk_n = end_idx - start_idx
            
            u_chunk = torch.randn(chunk_n, n_dims, device=device)
            u_chunk = u_chunk / torch.norm(u_chunk, p=2, dim=1, keepdim=True)
            
            samples_chunk = center_flat.unsqueeze(0) + u_chunk * scale.unsqueeze(0)
            
            lower_flat = torch.minimum(lower_flat, samples_chunk.min(dim=0).values)
            upper_flat = torch.maximum(upper_flat, samples_chunk.max(dim=0).values)
            
            del u_chunk, samples_chunk
        
        lower = lower_flat.view(center.shape)
        upper = upper_flat.view(center.shape)
    
    return LayerBounds(lower=lower, upper=upper)


def propagate_ellipsoid_affine(
    ellipsoid: Ellipsoid,
    W: torch.Tensor,
    b: Optional[torch.Tensor] = None,
    device: str = DEVICE
) -> Ellipsoid:
    """
    Propagate ellipsoid through affine transformation y = Wx + b.
    
    For diagonal P_in = diag(p_i):
        P_out = (W @ diag(1/p_i) @ W^T)^{-1}
    """
    center_in = ellipsoid.center
    P_in_diag = ellipsoid.P
    
    has_batch = P_in_diag.dim() > 1
    
    if has_batch:
        batch_size = P_in_diag.shape[0]
        if center_in.dim() > 2:
            center_in_flat = center_in.view(batch_size, -1)
        else:
            center_in_flat = center_in
        
        center_out_flat = F.linear(center_in_flat, W, bias=b)
        n_out = center_out_flat.shape[1]
        
        P_inv_diag = 1.0 / torch.clamp(P_in_diag, min=1e-8)
        
        P_out_list = []
        for b_idx in range(batch_size):
            P_inv_b = P_inv_diag[b_idx]
            W_scaled = W * P_inv_b.unsqueeze(0)
            W_Pinv_WT = W_scaled @ W.T
            
            # Use diagonal approximation for memory efficiency
            P_out_diag = 1.0 / torch.clamp(torch.diag(W_Pinv_WT), min=1e-8)
            P_out_list.append(P_out_diag)
        
        P_out = torch.stack(P_out_list)
        return Ellipsoid(center=center_out_flat, P=P_out, is_diagonal=True)
    else:
        if center_in.dim() > 1:
            center_in_flat = center_in.view(-1)
        else:
            center_in_flat = center_in
        
        center_out_flat = F.linear(center_in_flat.unsqueeze(0), W, bias=b).squeeze(0)
        
        P_inv_diag = 1.0 / torch.clamp(P_in_diag, min=1e-8)
        W_scaled = W * P_inv_diag.unsqueeze(0)
        W_Pinv_WT = W_scaled @ W.T
        
        P_out_diag = 1.0 / torch.clamp(torch.diag(W_Pinv_WT), min=1e-8)
        
        return Ellipsoid(center=center_out_flat, P=P_out_diag, is_diagonal=True)


def loewner_john_propagate_cnn3d(
    first_layer_bounds: LayerBounds,
    model: CNN3D,
    n_samples: int = 500,
    chunk_size: int = 100,
    device: str = DEVICE
) -> Tuple[LayerBounds, int]:
    """
    Propagate bounds through CNN3D using Löwner-John ellipsoid sampling.
    
    Args:
        first_layer_bounds: Bounds from first layer
        model: CNN3D model
        n_samples: Number of samples for ellipsoid
        chunk_size: Chunk size for memory efficiency
        device: Computation device
        
    Returns:
        (final_bounds, n_samples_used)
    """
    # Ensure batch dimension
    lower = first_layer_bounds.lower
    upper = first_layer_bounds.upper
    
    if lower.dim() == 4:
        lower = lower.unsqueeze(0)
        upper = upper.unsqueeze(0)
    
    lower = lower.to(device)
    upper = upper.to(device)
    
    # Convert to ellipsoid
    bounds = LayerBounds(lower=lower, upper=upper)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # Apply BN1 + ReLU (convert to bounds, apply, convert back)
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # Conv2 + BN2 (use IBP-style for conv, then re-ellipsoidize)
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    center = (temp_bounds.lower + temp_bounds.upper) / 2
    radius = (temp_bounds.upper - temp_bounds.lower) / 2
    
    w2_folded, b2_folded = get_folded_conv_params(model.conv2, model.bn2, device)
    
    center = F.conv3d(center, w2_folded, b2_folded,
                      stride=model.s2, padding=model.pd2)
    radius = F.conv3d(radius, w2_folded.abs(), None,
                      stride=model.s2, padding=model.pd2)
    
    lower_out = center - radius
    upper_out = center + radius
    bounds = LayerBounds(lower=lower_out, upper=upper_out)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # ReLU
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # Adaptive Average Pooling
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_pool = model.adaptive_pool(temp_bounds.lower)
    upper_pool = model.adaptive_pool(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_pool, upper=upper_pool)
    
    # Flatten
    bounds = LayerBounds(
        lower=bounds.lower.view(bounds.lower.size(0), -1),
        upper=bounds.upper.view(bounds.upper.size(0), -1)
    )
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # FC1
    ellipsoid = propagate_ellipsoid_affine(ellipsoid, model.fc1.weight, model.fc1.bias, device)
    
    # ReLU
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # FC2
    ellipsoid = propagate_ellipsoid_affine(ellipsoid, model.fc2.weight, model.fc2.bias, device)
    
    # ReLU
    temp_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    lower_tmp = F.relu(temp_bounds.lower)
    upper_tmp = F.relu(temp_bounds.upper)
    bounds = LayerBounds(lower=lower_tmp, upper=upper_tmp)
    ellipsoid = linf_bounds_to_ellipsoid(bounds, device)
    
    # FC3
    ellipsoid = propagate_ellipsoid_affine(ellipsoid, model.fc3.weight, model.fc3.bias, device)
    
    # Final sampling
    final_bounds = sample_ellipsoid_bounds(ellipsoid, n_samples, chunk_size, device)
    
    return final_bounds, n_samples

time: 9.16 ms (started: 2026-01-20 04:08:02 +00:00)


In [36]:
def closed_form_loewner_john_verify_cnn3d(
    model: CNN3D,
    input_tensor: torch.Tensor,
    epsilon: float,
    shared_frames: List[List[int]],
    spatial_mask: Optional[torch.Tensor] = None,
    n_samples: int = 500,
    chunk_size: int = 100,
    device: str = DEVICE
) -> Tuple[torch.Tensor, torch.Tensor, float, int]:
    """
    Complete verification: Closed-form first layer + Löwner-John ellipsoid.
    
    Args:
        model: CNN3D model
        input_tensor: Input tensor
        epsilon: Perturbation budget
        shared_frames: Frame groups with shared perturbations
        spatial_mask: Optional spatial mask
        n_samples: Number of ellipsoid samples
        chunk_size: Memory chunk size
        device: Computation device
        
    Returns:
        (lower_bounds, upper_bounds, runtime_seconds, n_samples_used)
    """
    start_time = time.time()
    
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    model = model.to(device)
    model.eval()
    input_tensor = input_tensor.to(device)
    
    # Step 1: Closed-form bounds on first Conv3D + BN1
    first_layer_bounds = closed_form_conv3d_bounds(
        input_tensor,
        model.conv1,
        epsilon,
        shared_frames,
        spatial_mask,
        model.bn1,
        device
    )
    
    # Step 2: Löwner-John propagation
    final_bounds, n_samples_used = loewner_john_propagate_cnn3d(
        first_layer_bounds,
        model,
        n_samples,
        chunk_size,
        device
    )
    
    runtime = time.time() - start_time
    
    return final_bounds.lower, final_bounds.upper, runtime, n_samples_used



time: 1.04 ms (started: 2026-01-20 04:11:06 +00:00)


In [51]:
closed_form_ljsamp_rob_acc = []
clean_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4]]


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.to(DEVICE).to(torch.long)
                        
        final_lower, final_upper, _, _ = closed_form_loewner_john_verify_cnn3d(
            model, X, epsilon=eps, shared_frames=frames, device=DEVICE, n_samples=50, chunk_size=10
        )
        output = model(X)
        y_pred = torch.argmax(output, dim=1)
        
        for i in range(len(X)):
            pred = y_pred[i]
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(truth_idx, pred)
            
            # final_lower, final_upper = interval_bound_propagation(
            #     model, X[i].unsqueeze(0), epsilon=val, lower_input=None, upper_input=None
            # )
            # pred = y_pred[i]
            # is_robust = ( final_lower[0][pred] > torch.cat((final_upper[0][:pred], final_upper[0][pred + 1:], ))).all().item()
            # is_classification_correct = torch.equal(y[i], pred)
            
            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    

    clean_acc.append((correct / total) * 100)
    closed_form_ljsamp_rob_acc.append((robust / total) * 100)
    print(f"Epsilon: {eps} => Clean Acc: {clean_acc[-1]:.2f}%, Closed-form-Löwner-John Sampling Robust Acc: {closed_form_ljsamp_rob_acc[-1]:.2f}%")




print("Model Accuracy: ", clean_acc)
print("Closed-form-Löwner-John Sampling Robust Accuracy: ", closed_form_ljsamp_rob_acc)

# print(correct," ",robust, " ",total)

Epsilon: 0.1 => Clean Acc: 84.24%, Closed-form-Löwner-John Sampling Robust Acc: 84.24%
Epsilon: 0.01 => Clean Acc: 84.24%, Closed-form-Löwner-John Sampling Robust Acc: 84.24%
Epsilon: 0.001 => Clean Acc: 84.24%, Closed-form-Löwner-John Sampling Robust Acc: 84.24%
Epsilon: 0.0001 => Clean Acc: 84.24%, Closed-form-Löwner-John Sampling Robust Acc: 84.24%
Epsilon: 1e-05 => Clean Acc: 84.24%, Closed-form-Löwner-John Sampling Robust Acc: 84.24%
Model Accuracy:  [84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579]
Closed-form-Löwner-John Sampling Robust Accuracy:  [84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579, 84.23864648263579]
time: 11min 13s (started: 2026-01-20 05:12:19 +00:00)


#### Selective Interval Bound Propagation - Adversarial Patches

For the adversarial patches, we implement the closed form solution with Lipschitz bounds propagation.

In [12]:
def closed_form_lipschitz_verify_patch(
    model: CNN3D,
    input_tensor: torch.Tensor,
    epsilon: float,
    frame_patch_locations: Dict[int, List[Tuple[int, int, int, int]]],
    device: str = DEVICE
) -> Tuple[torch.Tensor, torch.Tensor, float, float]:
    """
    Complete verification pipeline for adversarial patch constraints:
    Closed form Conv3D bounds (patch) + IBP propagation.
    
    This combines:
    1. Closed form bounds on first Conv3D layer for patch perturbations (EXACT)
    2. IBP propagation through remaining layers (SOUND, L∞)
    
    Args:
        model: Simple3DCNN model
        input_tensor: Input tensor (C, D, H, W) or (B, C, D, H, W)
        epsilon: Perturbation budget (L∞ norm)
        frame_patch_locations: Dictionary mapping frame index to patch bounding boxes
        device: Computation device
        
    Returns:
        (lower_bounds, upper_bounds, runtime_seconds, spectral_norm_diagnostic)
    """
    start_time = time.time()
    
    # Ensure input has correct shape
    if input_tensor.dim() == 4:
        input_tensor = input_tensor.unsqueeze(0)
    
    # Move model and input to device
    model = model.to(device)
    model.eval()
    input_tensor = input_tensor.to(device)
    
    # Step 1: Analytical bounds on first Conv3D layer with patch constraints (EXACT)
    first_layer_bounds = closed_form_conv3d_bounds_patch(
        input_tensor,
        model.conv1,
        epsilon,
        frame_patch_locations,
        device
    )

    # Step 2: Lipschitz propagation through remaining layers (SOUND)
    final_bounds, spectral_norm = lipschitz_propagate_cnn3d(
        first_layer_bounds,
        model,
        device
    )
    
    runtime = time.time() - start_time
    
    return final_bounds.lower, final_bounds.upper, runtime, spectral_norm

time: 1.58 ms (started: 2026-03-30 14:41:26 +01:00)


In [13]:
def process_bounding_boxes_udacity(
     bounding_box_info: Dict, H: int, W: int, filename: str
) -> Dict[int, List[Tuple[int, int, int, int]]]:
    """
    Process bounding box information into frame_patch_locations format.

    Args:
        bounding_box_info: Dictionary containing bounding box information for each object
        H: Height of the video frames
        W: Width of the video frames
        filename: Name of the image file

    Returns:
        Dictionary with frame indices as keys and lists of (x1, y1, x2, y2) tuples as values
    """
    frame_patch_locations = {}

    # Get the frame number from image name (e.g., "1479425441432724303.jpg" -> 1479425441432724303)
    frame_num = filename
    # Process each object's bounding boxes
    for (x1, y1, x2, y2) in bounding_box_info["xyxyn"]:
        # Get normalized bounding box coordinates

        # Convert normalized coordinates to pixel coordinates
        x1_px = int(x1 * W)
        y1_px = int(y1 * H)
        x2_px = int(x2 * W)
        y2_px = int(y2 * H)

        # Store the coordinates in frame_patch_locations as a list
        if frame_num not in frame_patch_locations:
            frame_patch_locations[frame_num] = []
        frame_patch_locations[frame_num].append((x1_px, y1_px, x2_px, y2_px))

    return frame_patch_locations




time: 1.18 ms (started: 2026-03-30 14:41:30 +01:00)


In [14]:
import json
def get_bounding_box_info_udacity(bounding_box_path: str, json_file_name: str) -> Dict:
    """
    Load bounding box information from a JSON file.

    Args:
        bounding_box_path: Path to the JSON file containing bounding box information

    Returns:
        Dictionary containing bounding box information
    """
    with open(os.path.join(bounding_box_path, json_file_name), "r") as f:
        bounding_box_info = json.load(f)
    return bounding_box_info

time: 733 μs (started: 2026-03-30 14:41:32 +01:00)


In [53]:
start_time = time.time()
sibp_patch_rob_acc = []
epsilons = [0.00001, 0.0001, 0.001, 0.01, 0.1]
for epsilon in epsilons:
    for batch_idx, (x, y) in enumerate(valid_loader):  
        x = x.to(device)
        y = y.to(device)
        verified_correct = 0
        total = x.size(0)

        # print(f"\nIBP Verification with epsilon={epsilon}")
        # print(f"Input shape: {x.shape}")
        # print(f"Number of samples: {total}")

        for i in range(total):
            # print(f"\nVerifying sample {i+1}/{total}")
            input_tensor = x[i:i+1]
            y_i = y[i:i+1]
        
            with torch.no_grad():
                file_name = valid_loader.dataset.dataset.data_frame.loc[batch_idx*batch_size + i, 'frame_id']
                bounding_box_info = get_bounding_box_info_udacity(
                    BOUNDING_BOX_PATH, f"{file_name}_tracked.json"
                )
                frame_patch_locations = process_bounding_boxes_udacity(
                    bounding_box_info, input_tensor.shape[3], input_tensor.shape[4], file_name
                )
                # print(bounding_box_info)
                lower_bounds, upper_bounds, _, _ = closed_form_lipschitz_verify_patch(
                    model, input_tensor, epsilon=epsilon, frame_patch_locations=frame_patch_locations, device=DEVICE
                )
            
            # Print bound statistics
            # print(f"Lower bounds range: [{lower_bounds.min().item():.4f}, {lower_bounds.max().item():.4f}]")
            # print(f"Upper bounds range: [{upper_bounds.min().item():.4f}, {upper_bounds.max().item():.4f}]")
            # print(f"Bound width: {(upper_bounds - lower_bounds).mean().item():.4f}")

            # Check robustness
            margin = lower_bounds[0, y_i] - torch.max(
                torch.cat([upper_bounds[0, :y_i], upper_bounds[0, y_i + 1 :]], dim=0)
            )

            if margin.item() > 0:
                verified_correct += 1
                # print(f"Sample {i+1} is robust (margin: {margin.item():.4f})")
            # else:
            #     print(f"Sample {i+1} is not robust (margin: {margin.item():.4f})")

    sibp_patch_rob_acc.append(verified_correct / total)
print(f"Total time taken: {time.time() - start_time:.2f} seconds")

Total time taken: 65.97 seconds
time: 1min 5s (started: 2026-01-20 05:30:38 +00:00)


In [54]:
sibp_patch_rob_acc

[0.8, 0.8, 0.8, 0.8, 0.8]

time: 6.22 ms (started: 2026-01-20 05:51:06 +00:00)


In [ ]:
closed_form_patch_rob_acc = []
closed_form_patch_acc = []

epsilons = [3, 2.75, 2.50, 2.25, 2, 1.75, 1.5, 1.25, 1, 0.75, 0.5, 0.25, 0.1, 0.075, 0.05, 0.025, 0.01, 0.0075, 0.005, 0.0025, 0.001, 0.0001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4]]
input_frames = 30

# Bounding box spec
bb_spec = {
    "image": "sample_image",
    "xyxyn": [

            0.2983497381210327,
            0.3809495419217273593,
            0.3970875978469849,
            0.4800417633056641
        ]
}

x1n, y1n, x2n, y2n = bb_spec["xyxyn"]
 # Convert normalized coordinates to pixel coordinates
x1 = int(x1n * resize_shape[0])
y1 = int(y1n * resize_shape[1])
x2 = int(x2n * resize_shape[0])
y2 = int(y2n * resize_shape[1])

# Define patch specification (64 x 64) - approx 6 x 6 patch
patch_spec = {
    'patch_size': (6, 6),
    'frame_patch_locations': {
        i: (x1, y1, x2, y2) for i in range(input_frames)
    }
}


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.squeeze().to(DEVICE).to(torch.long)
        
        final_lower, final_upper, _, _ = closed_form_ibp_verify_patch(
            model, X, epsilon=eps, frame_patch_locations=patch_spec['frame_patch_locations'], device=DEVICE
        )
        
        y_pred = model(X)
        for i in range(len(X)):
            
            pred = torch.argmax(y_pred[i])
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(y[i], pred)

            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    closed_form_patch_acc.append((correct / total) * 100)
    closed_form_patch_rob_acc.append((robust / total) * 100)



print("Model Accuracy: ", closed_form_patch_acc)
print("Closed Form Lipschitz Patch Robust Accuracy: ", closed_form_patch_rob_acc)


Model Accuracy:  [72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339, 72.88135593220339]
Closed Form Lipschitz Patch Robust Accuracy:  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 33.33333333333333, 72.88135593220339]
time: 2min 6s (started: 2026-01-19 22:05:36 +00:00)


In [ ]:
closed_form_patch_rob_acc = []
closed_form_patch_acc = []

epsilons = [0.1, 0.01, 0.001, 0.0001, 0.00001]

# Generate constraints and perturbation tensor - Keeping shared perturbations same for all epsilons
frames = [[0,1,2,3,4]]
input_frames = 30

# Bounding box spec
bb_spec = {
    "image": "sample_image",
    "xyxyn": [

            0.2983497381210327,
            0.3809495419217273593,
            0.3970875978469849,
            0.4800417633056641
        ]
}

x1n, y1n, x2n, y2n = bb_spec["xyxyn"]
 # Convert normalized coordinates to pixel coordinates
x1 = int(x1n * resize_shape[0])
y1 = int(y1n * resize_shape[1])
x2 = int(x2n * resize_shape[0])
y2 = int(y2n * resize_shape[1])

# Define patch specification (64 x 64) - approx 6 x 6 patch
patch_spec = {
    'patch_size': (6, 6),
    'frame_patch_locations': {
        i: (x1, y1, x2, y2) for i in range(input_frames)
    }
}


for eps in epsilons:
    robust = 0
    correct = 0
    total = 0
    for idx, (X, y) in enumerate(val_loader):
        X = X.to(DEVICE)
        y = y.squeeze().to(DEVICE).to(torch.long)
        
        final_lower, final_upper, _, _ = closed_form_ibp_verify_patch(
            model, X, epsilon=eps, frame_patch_locations=patch_spec['frame_patch_locations'], device=DEVICE
        )
        
        y_pred = model(X)
        for i in range(len(X)):
            
            pred = torch.argmax(y_pred[i])
            truth_idx = y[i]
            is_robust = ( final_lower[i][truth_idx] > torch.cat((final_upper[i][:truth_idx], final_upper[i][truth_idx + 1:], ))).all().item()
            is_classification_correct = torch.equal(y[i], pred)

            robust += 1 if is_robust else 0
            correct += 1 if is_classification_correct else 0
            total += 1
    closed_form_patch_acc.append((correct / total) * 100)
    closed_form_patch_rob_acc.append((robust / total) * 100)



print("Model Accuracy: ", closed_form_patch_acc)
print("Closed Form Lipschitz Patch Robust Accuracy: ", closed_form_patch_rob_acc)


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

time: 2.14 s (started: 2026-03-30 14:46:26 +01:00)


In [21]:
sibp_patch_rob_acc

[0.8571428571428571,
 0.8571428571428571,
 0.8571428571428571,
 0.8571428571428571,
 0.8571428571428571]